
<div style="background:linear-gradient(135deg,#0d1b2a 0%,#1b2838 60%,#243447 100%);padding:48px 40px;border-radius:16px;font-family:'Georgia',serif;color:white;text-align:center;border:1px solid #2e4a6a">
  <div style="font-size:11px;letter-spacing:6px;text-transform:uppercase;color:#7eb3d8;margin-bottom:16px">FIFA · MUNDIAL 2026</div>
  <h1 style="font-size:38px;font-weight:700;margin:0 0 10px;line-height:1.2">Modelo Predictivo XGBoost</h1>
  <div style="font-size:16px;color:#a8c4d8;margin-bottom:28px">Clasificación ordinal de 48 selecciones · max_depth=5</div>
  <hr style="border:none;border-top:1px solid #2e4a6a;margin:24px 0">
  <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:16px;max-width:700px;margin:0 auto">
    <div style="background:rgba(255,255,255,.06);border-radius:10px;padding:14px 8px">
      <div style="font-size:22px;font-weight:700;color:#60b8f5">48</div>
      <div style="font-size:11px;color:#7eb3d8;margin-top:4px">Selecciones</div>
    </div>
    <div style="background:rgba(255,255,255,.06);border-radius:10px;padding:14px 8px">
      <div style="font-size:22px;font-weight:700;color:#60b8f5">7</div>
      <div style="font-size:11px;color:#7eb3d8;margin-top:4px">Variables</div>
    </div>
    <div style="background:rgba(255,255,255,.06);border-radius:10px;padding:14px 8px">
      <div style="font-size:22px;font-weight:700;color:#60b8f5">128</div>
      <div style="font-size:11px;color:#7eb3d8;margin-top:4px">Muestras train</div>
    </div>
    <div style="background:rgba(255,255,255,.06);border-radius:10px;padding:14px 8px">
      <div style="font-size:22px;font-weight:700;color:#60b8f5">5</div>
      <div style="font-size:11px;color:#7eb3d8;margin-top:4px">max_depth</div>
    </div>
  </div>
</div>


## 📋 Índice

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | Instalación | Librerías |
| 2 | Dataset v6 | 48 equipos · tm_cat · wc_score · copa_score |
| 3 | Datos históricos | Solo WC2022 + WC2018 (64 muestras) |
| 4 | Feature Engineering | Escala diferenciada · gain controlado |
| 5 | Entrenamiento XGBoost Regresor | MAE · sample_weight · LOO-CV |
| 6 | Predicción 2026 | Posiciones 1–48 · fase_idx |
| 7–12 | Dashboards | Ranking · FI · Grupos · Radar · Bubble · Heatmap |
| 13 | Fixture Grupos | Regresor + score XGBoost + MAE como σ |
| 14 | **Bracket Eliminatorias** | **Clasificador Binario + Fixture Oficial FIFA** |



---
## 1. Instalación y dependencias

Se requieren las siguientes librerías. Si estás en un entorno limpio, ejecuta la celda de instalación.


In [1]:
# ── Instalación (descomentar si es necesario) ─────────────────────────────────
# !pip install xgboost scikit-learn pandas numpy plotly

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json

print("✅ Todas las librerías cargadas correctamente")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


✅ Todas las librerías cargadas correctamente
   pandas  2.3.3
   numpy   2.4.0


---
## 2. Dataset — 48 clasificados 2026 (v8)

### Variables del modelo

| Variable | Descripción | Fuente |
|----------|-------------|--------|
| `wc_score` | WC2022×2.0 + WC2018×1.5 — posición invertida | FIFA |
| `elim_fuerza` | Fuerza Final en eliminatorias (Fuerza Base − Penalidad) | CSV Eliminatorias |
| `copa_poder` | Poder Final en copas intercontinentales (Fuerza Base + Ajuste conf.) | CSV Copas |
| `ranking_score` | 210 − Ranking FIFA | FIFA |
| `tm_cat` | Categoría Transfermarkt (0=<100M · 1=100-400M · 2=400-650M · 3=>650M) | Transfermarkt |
| `copa_score` | Rendimiento copa continental × dificultad confederación | Fuentes oficiales |
| `rating_eafc26` | EA FC 26 squad overall (÷10 para reducir gain) | EA Sports |
| `primera_vez` | 1 = debut absoluto → forzado 4to de grupo | FIFA |

### Por qué separar `elim_fuerza` y `copa_poder`

El `torneo_score` combinado (CWC×2.8 + FIC×2.2 + Elim×1.8) tenía gain bajo (0.047)
porque promediaba señales de diferente naturaleza. Al separarlas:
- **`elim_fuerza`**: captura la dificultad real del camino clasificatorio (rivales directos, penalidad por posición final)
- **`copa_poder`**: captura el rendimiento reciente en copas (ajuste por confederación ya incluido)

Ambas vienen directamente de los CSVs proporcionados, con fórmulas propias.

### Valores por defecto para equipos sin datos en CSV

| Equipo | elim_fuerza | copa_poder | Razón |
|--------|-------------|-----------|-------|
| Sudáfrica | 105.0 | 155.7 ✓ | No en CSV elim → media CAF estimada |
| Chequia | 104.6 | 154.1 ✓ | No en CSV elim → estimado por vía playoff |
| Cabo Verde | 90.0 | 90.0 | Debut; muy por debajo de la media |
| DR Congo | 95.0 | 153.1 ✓ | No en CSV elim → estimado CAF |
| Bosnia-Herz. | 104.6 ✓ | 100.0 | No en CSV copa → estimado bajo |
| Haití | 110.0 ✓ | 95.0 | No en CSV copa → estimado CONCACAF bajo |
| Curazao | 110.0 ✓ | 90.0 | No en CSV copa → debut internacional |
| Ghana | 110.0 ✓ | 100.0 | No en CSV copa → estimado CAF |


In [2]:
# ── Dataset v8 — variables reales desde CSV ──────────────────────────────────
# Fuentes:
#   elim_fuerza : CSV Eliminatorias — columna "Fuerza Final"
#   copa_poder  : CSV Copas_intercontinentales — columna "Poder Final"
# Valores basados en fórmulas propias del usuario (dificultad de rivales,
# penalidades por posición, ajuste de confederación, etc.)

CONF_WEIGHTS = {
    "UEFA":0.96,"CONMEBOL":0.91,"CAF":0.71,
    "AFC":0.55,"CONCACAF":0.48,"OFC":0.11,
}

def tm_cat(val_m):
    if val_m < 100:  return 0
    if val_m < 400:  return 1
    if val_m < 650:  return 2
    return 3

def copa_score_v8(tit_rec, pos_rec, tit_ant, pos_ant, conf):
    def raw(t, p):
        if p >= 99: return t * 0.5
        return (t * 1.0 + max(0, 1 - (p-1)/12) * 0.6) / 1.6
    s = (raw(tit_rec, pos_rec) * 2.0 + raw(tit_ant, pos_ant) * 1.0) / 3.0
    return round(s * CONF_WEIGHTS[conf], 4)

def wc_score_v8(p22, p18):
    s22 = (1 - (p22-1)/47) * 2.0 if p22 < 99 else 0.0
    s18 = (1 - (p18-1)/31) * 1.5 if p18 < 99 else 0.0
    return round(s22 + s18, 4)

# ── Datos completos de los 48 equipos ────────────────────────────────────────
# equipo, grupo, ranking_fifa, rating_eafc26, primera_vez,
# tm_value_m, tit_rec, pos_rec, tit_ant, pos_ant, conf,
# pos_wc22, pos_wc18,
# elim_fuerza (CSV), copa_poder (CSV)

RAW_2026 = [
    # GRUPO A
    ("México",            "A", 15, 77, 0,  150, 1, 1, 1, 2, "CONCACAF", 17, 17, 125.95, 171.65),
    ("Corea del Sur",     "A", 25, 72, 0,  140, 0, 3, 0, 4, "AFC",       9, 17, 112.00, 167.90),
    ("Sudáfrica",         "A", 60, 40, 0,   40, 0, 4, 0, 4, "CAF",      99, 99, 105.00, 155.70),
    ("Chequia",           "A", 41, 66, 0,  120, 0,13, 0, 5, "UEFA",     99, 99, 104.60, 154.10),
    # GRUPO B
    ("Canadá",            "B", 30, 70, 0,  120, 0, 4, 0,13, "CONCACAF", 17, 99, 125.95, 174.35),
    ("Bosnia-Herzegovina","B", 35, 66, 0,   95, 0,99, 0,99, "UEFA",     99, 99, 104.60, 100.00),
    ("Qatar",             "B", 56, 59, 0,   55, 2, 1, 2, 1, "AFC",      17, 99, 107.00, 171.30),
    ("Suiza",             "B", 19, 74, 0,  240, 0, 5, 0, 5, "UEFA",      9, 17, 151.00, 169.90),
    # GRUPO C
    ("Brasil",            "C",  6, 83, 0, 1100, 0, 5, 0, 2, "CONMEBOL",  5, 17, 157.55, 171.45),
    ("Marruecos",         "C",  8, 79, 0,  270, 1, 1, 0,13, "CAF",       4, 17, 115.00, 158.45),
    ("Haití",             "C", 85, 54, 1,    8, 0,13, 0,13, "CONCACAF", 99, 99, 110.00,  95.00),
    ("Escocia",           "C", 43, 66, 0,   90, 0,13, 0,13, "UEFA",     99, 99, 149.25, 156.40),
    # GRUPO D
    ("Estados Unidos",    "D", 16, 74, 0,  420, 0,13, 1, 1, "CONCACAF",  9, 17, 125.95, 155.75),
    ("Paraguay",          "D", 40, 65, 0,   85, 0,13, 0,13, "CONMEBOL", 17, 17, 159.33, 168.00),  # copa_poder +10 ajuste QF
    ("Australia",         "D", 27, 70, 0,  120, 0, 5, 0,13, "AFC",       9, 17, 112.00, 164.00),
    ("Turquía",           "D", 22, 72, 0,  210, 0, 5, 0, 9, "UEFA",     17, 17, 152.50, 157.90),
    # GRUPO E
    ("Alemania",          "E", 10, 83, 0,  950, 0, 5, 0, 9, "UEFA",     17, 17, 147.00, 172.90),
    ("Curazao",           "E",110, 50, 1,   12, 0,13, 0,13, "CONCACAF", 99, 99, 110.00,  90.00),
    ("Costa de Marfil",   "E", 34, 69, 0,  140, 1, 1, 0, 9, "CAF",      99, 99, 110.00, 165.75),
    ("Ecuador",           "E", 23, 70, 0,  100, 0, 5, 0,13, "CONMEBOL", 17, 99, 164.88, 169.00),
    # GRUPO F
    ("Países Bajos",      "F",  7, 83, 0,  820, 0, 3, 0, 9, "UEFA",      5, 17, 117.80, 181.10),
    ("Japón",             "F", 18, 73, 0,  260, 0, 5, 0, 2, "AFC",       9, 17, 112.00, 166.45),
    ("Suecia",            "F", 38, 78, 0,  160, 0,13, 0, 9, "UEFA",     99,  5, 145.00, 153.80),  # WC2018 QF (pos=5)
    ("Túnez",             "F", 44, 63, 0,   60, 0,13, 0,13, "CAF",      17, 17, 115.00, 151.95),
    # GRUPO G
    ("Bélgica",           "G",  9, 78, 0,  580, 0,13, 0, 5, "UEFA",     17, 17, 117.80, 173.40),
    ("Egipto",            "G", 29, 72, 0,   70, 0,13, 0, 2, "CAF",      99, 99, 110.00, 160.90),
    ("Irán",              "G", 21, 69, 0,   55, 0, 3, 0,13, "AFC",      17, 17, 112.00, 170.00),
    ("Nueva Zelanda",     "G", 95, 55, 0,   20, 2, 1, 2, 1, "OFC",      99, 99,  55.00, 140.00),  # copa_poder -20 OFC
    # GRUPO H
    ("España",            "H",  2, 85, 0, 1050, 1, 1, 0, 3, "UEFA",     17,  9, 155.50, 184.60),
    ("Cabo Verde",        "H", 69, 57, 1,   15, 0, 9, 0,13, "CAF",      99, 99,  90.00,  90.00),
    ("Arabia Saudita",    "H", 61, 60, 0,   65, 0, 9, 0,13, "AFC",      17, 17, 107.00, 153.50),
    ("Uruguay",           "H", 17, 76, 0,  320, 0, 3, 0, 4, "CONMEBOL", 17, 17, 160.77, 172.90),
    # GRUPO I
    ("Francia",           "I",  1, 85, 0, 1280, 0, 3, 0, 9, "UEFA",      2,  1, 146.25, 181.45),
    ("Senegal",           "I", 14, 73, 0,  280, 0, 2, 1, 1, "CAF",      17, 17, 115.00, 161.25),
    ("Iraq",              "I", 57, 58, 0,   22, 0, 9, 0,13, "AFC",      99, 99,  97.00, 157.00),
    ("Noruega",           "I", 31, 78, 0,  510, 0,99, 0,99, "UEFA",     99, 99, 125.20, 155.25),
    # GRUPO J
    ("Argentina",         "J",  3, 83, 0,  620, 2, 1, 2, 1, "CONMEBOL",  1, 17, 165.22, 177.00),
    ("Argelia",           "J", 28, 70, 0,   90, 0,13, 0,13, "CAF",      99, 99, 115.00, 154.75),  # no fue WC2018 ni WC2022
    ("Austria",           "J", 24, 72, 0,  220, 0, 9, 0, 9, "UEFA",     99, 99, 107.60, 159.30),
    ("Jordania",          "J", 63, 55, 1,   18, 0, 2, 0,13, "AFC",      99, 99, 112.00, 157.35),
    # GRUPO K
    ("Portugal",          "K",  5, 84, 0,  780, 0, 5, 0, 9, "UEFA",      9, 17, 151.50, 175.06),
    ("DR Congo",          "K", 46, 60, 0,   65, 0, 3, 0,13, "CAF",      99, 99,  95.00, 120.00),  # copa_poder ajustado
    ("Uzbekistán",        "K", 50, 57, 1,   45, 0, 5, 0,13, "AFC",      99, 99, 112.00, 161.55),
    ("Colombia",          "K", 13, 76, 0,  380, 0, 2, 0, 3, "CONMEBOL", 17, 17, 162.33, 175.65),
    # GRUPO L
    ("Inglaterra",        "L",  4, 84, 0, 1400, 0, 2, 0, 2, "UEFA",      9,  4, 119.80, 175.00),
    ("Croacia",           "L", 11, 79, 0,  380, 0,13, 0, 9, "UEFA",      3,  2, 129.20, 168.15),
    ("Ghana",             "L", 55, 64, 0,   75, 0,13, 0,13, "CAF",      17, 17, 110.00, 100.00),
    ("Panamá",            "L", 33, 62, 0,   30, 0, 2, 0,13, "CONCACAF", 17, 17, 110.00, 162.60),
]

COLS = ["equipo","grupo","ranking_fifa","rating_eafc26","primera_vez",
        "tm_value_m","tit_rec","pos_rec","tit_ant","pos_ant","conf",
        "pos_wc22","pos_wc18","elim_fuerza","copa_poder"]

df_2026 = pd.DataFrame(RAW_2026, columns=COLS)

# ── Feature engineering ───────────────────────────────────────────────────────
df_2026["ranking_score"] = 210 - df_2026["ranking_fifa"]
df_2026["tm_cat"]        = df_2026["tm_value_m"].apply(tm_cat)
df_2026["copa_score"]    = df_2026.apply(
    lambda r: copa_score_v8(r.tit_rec, r.pos_rec, r.tit_ant, r.pos_ant, r.conf), axis=1)
df_2026["wc_score"]      = df_2026.apply(
    lambda r: wc_score_v8(r.pos_wc22, r.pos_wc18), axis=1)

# Normalizar elim_fuerza y copa_poder al rango [0,1] para comparabilidad
ef_min, ef_max = 55.0, 170.0
cp_min, cp_max = 90.0, 185.0
df_2026["elim_norm"] = (df_2026["elim_fuerza"] - ef_min) / (ef_max - ef_min)
df_2026["copa_norm"]  = (df_2026["copa_poder"]  - cp_min) / (cp_max - cp_min)

print(f"✅ Dataset v8: {len(df_2026)} equipos")
print()
print("Top 10 — elim_fuerza (dificultad eliminatorias):")
print(df_2026[["equipo","elim_fuerza","elim_norm"]].sort_values("elim_fuerza",ascending=False).head(10).to_string(index=False))
print()
print("Top 10 — copa_poder (rendimiento copas):")
print(df_2026[["equipo","copa_poder","copa_norm"]].sort_values("copa_poder",ascending=False).head(10).to_string(index=False))


✅ Dataset v8: 48 equipos

Top 10 — elim_fuerza (dificultad eliminatorias):
   equipo  elim_fuerza  elim_norm
Argentina       165.22   0.958435
  Ecuador       164.88   0.955478
 Colombia       162.33   0.933304
  Uruguay       160.77   0.919739
 Paraguay       159.33   0.907217
   Brasil       157.55   0.891739
   España       155.50   0.873913
  Turquía       152.50   0.847826
 Portugal       151.50   0.839130
    Suiza       151.00   0.834783

Top 10 — copa_poder (rendimiento copas):
      equipo  copa_poder  copa_norm
      España      184.60   0.995789
     Francia      181.45   0.962632
Países Bajos      181.10   0.958947
   Argentina      177.00   0.915789
    Colombia      175.65   0.901579
    Portugal      175.06   0.895368
  Inglaterra      175.00   0.894737
      Canadá      174.35   0.887895
     Bélgica      173.40   0.877895
     Uruguay      172.90   0.872632



---
## 3. Datos históricos de entrenamiento

Usamos los 4 últimos Mundiales como conjunto de entrenamiento (128 muestras).
La variable target es la **posición final** real de cada selección.

Las variables históricas se construyen con la misma lógica que los datos de 2026,
usando datos disponibles al momento de cada torneo.


In [3]:
# ── Datos históricos Mundiales 2010, 2014, 2018, 2022 ────────────────────────
# Columnas: posicion_final, ranking_score, rating_approx, primera_vez,
#           pos_prev_mundial, elim_pct, copa_titulos_ult2, copa_mejor_pos_ult2

HIST = [
    # ── SUDÁFRICA 2010 ───────────────────────────────────────────────────────
    (1,  208, 88, 0,  1, 0.17, 1, 1),  # España
    (2,  206, 86, 0,  2, 0.22, 0, 2),  # Países Bajos
    (3,  204, 87, 0,  3, 0.14, 1, 3),  # Alemania
    (4,  194, 82, 0,  4, 0.25, 0, 2),  # Uruguay
    (5,  203, 91, 0,  5, 0.10, 1, 1),  # Argentina
    (5,  209, 89, 0,  1, 0.10, 2, 1),  # Brasil
    (5,  178, 76, 0, 17, 0.40, 0, 5),  # Ghana
    (5,  179, 74, 0, 17, 0.38, 0, 5),  # Paraguay
    (9,  202, 84, 0,  9, 0.17, 0, 5),  # Inglaterra
    (9,  193, 79, 0, 17, 0.25, 1, 2),  # México
    (9,  163, 74, 0, 17, 0.50, 0, 9),  # Corea del Sur
    (9,  196, 78, 0,  9, 0.22, 1, 3),  # Estados Unidos
    (9,  165, 73, 0, 17, 0.60, 0, 9),  # Japón
    (9,  192, 78, 0, 17, 0.28, 0, 5),  # Chile
    (9,  207, 88, 0,  9, 0.12, 0, 5),  # Portugal
    (9,  176, 70, 0, 17, 0.43, 0, 9),  # Eslovaquia
    (17, 201, 83, 0,  1, 0.17, 1, 3),  # Francia
    (17, 205, 85, 0,  1, 0.14, 1, 1),  # Italia
    (17, 187, 74, 0, 17, 0.33, 0, 9),  # Grecia
    (17, 189, 75, 0, 17, 0.38, 0, 9),  # Nigeria
    (17, 199, 72, 0, 17, 0.42, 0,13),  # Camerún
    (17, 183, 80, 0, 17, 0.35, 0, 9),  # Costa de Marfil
    (17, 190, 74, 0, 17, 0.30, 0,13),  # Serbia
    (17, 174, 73, 0, 17, 0.45, 0, 9),  # Dinamarca
    (17, 187, 75, 0, 17, 0.28, 0, 9),  # Suiza
    (17, 172, 67, 0, 33, 0.60, 0,13),  # Honduras
    (17, 180, 68, 0, 33, 0.55, 0,13),  # Argelia
    (17, 185, 70, 0, 17, 0.40, 0, 9),  # Eslovenia
    (17, 190, 72, 0, 17, 0.35, 0,13),  # Australia
    (17, 132, 63, 0, 17, 0.78, 0,13),  # Nueva Zelanda
    (17, 105, 55, 0, 33, 0.90, 0,13),  # Corea del Norte
    (17, 170, 67, 0, 17, 0.50, 0, 9),  # Costa Rica
    # ── BRASIL 2014 ──────────────────────────────────────────────────────────
    (1,  208, 89, 0,  3, 0.14, 1, 1),  # Alemania
    (2,  205, 91, 0,  1, 0.10, 0, 1),  # Argentina
    (3,  202, 87, 0,  2, 0.22, 0, 2),  # Países Bajos
    (4,  206, 86, 0,  1, 0.10, 1, 1),  # Brasil
    (5,  201, 85, 0,  5, 0.20, 0, 3),  # Colombia
    (5,  177, 73, 0, 17, 0.38, 0, 9),  # Costa Rica
    (5,  184, 85, 0,  1, 0.17, 1, 3),  # Francia
    (5,  199, 85, 0,  5, 0.22, 0, 3),  # Bélgica
    (9,  198, 83, 0,  9, 0.20, 0, 5),  # Chile
    (9,  186, 79, 0, 17, 0.30, 0, 9),  # México
    (9,  204, 80, 0, 17, 0.20, 0, 5),  # Suiza
    (9,  187, 75, 0, 17, 0.35, 0, 9),  # Grecia
    (9,  194, 78, 0,  9, 0.22, 1, 3),  # Estados Unidos
    (9,  180, 75, 0, 17, 0.38, 0, 9),  # Argelia
    (9,  167, 75, 0, 17, 0.45, 0, 9),  # Nigeria
    (9,  204, 84, 0,  4, 0.12, 0, 5),  # Uruguay
    (17, 209, 89, 0,  1, 0.14, 1, 1),  # España
    (17, 191, 83, 0,  1, 0.17, 1, 3),  # Italia
    (17, 200, 82, 0,  9, 0.17, 0, 5),  # Inglaterra
    (17, 207, 88, 0,  9, 0.12, 0, 3),  # Portugal
    (17, 177, 73, 0, 17, 0.33, 0, 9),  # Ecuador
    (17, 163, 74, 0, 17, 0.50, 0, 9),  # Ghana
    (17, 164, 73, 0, 17, 0.55, 0, 9),  # Japón
    (17, 191, 78, 0, 17, 0.28, 0, 9),  # Croacia
    (17, 153, 70, 0, 17, 0.60, 0,13),  # Camerún
    (17, 177, 65, 0, 33, 0.62, 0,13),  # Honduras
    (17, 167, 70, 0, 17, 0.50, 0, 9),  # Irán
    (17, 192, 77, 0, 17, 0.30, 0, 9),  # Bosnia
    (17, 151, 70, 0, 33, 0.65, 0,13),  # Australia
    (17, 153, 71, 0, 17, 0.58, 0, 9),  # Corea del Sur
    (17, 191, 75, 0, 17, 0.28, 0, 9),  # Rusia
    (17, 187, 79, 0, 17, 0.35, 0, 9),  # Costa de Marfil
    # ── RUSIA 2018 ───────────────────────────────────────────────────────────
    (1,  203, 87, 0,  1, 0.17, 1, 1),  # Francia
    (2,  190, 84, 0,  3, 0.20, 0, 3),  # Croacia
    (3,  207, 87, 0,  5, 0.17, 0, 3),  # Bélgica
    (4,  198, 84, 0,  9, 0.17, 0, 3),  # Inglaterra
    (5,  196, 84, 0,  4, 0.20, 0, 3),  # Uruguay
    (5,  208, 88, 0,  1, 0.10, 1, 1),  # Brasil
    (5,  185, 76, 0, 17, 0.38, 0, 5),  # Suecia
    (5,  140, 71, 0, 17, 0.58, 0, 9),  # Rusia
    (9,  205, 90, 0,  1, 0.12, 0, 2),  # Argentina
    (9,  206, 87, 0,  1, 0.12, 1, 3),  # Portugal
    (9,  201, 87, 0,  1, 0.14, 1, 1),  # España
    (9,  191, 77, 0, 17, 0.30, 0, 5),  # Dinamarca
    (9,  149, 74, 0, 17, 0.55, 0, 9),  # Japón
    (9,  195, 80, 0, 17, 0.22, 1, 2),  # México
    (9,  194, 82, 0,  5, 0.22, 0, 3),  # Colombia
    (9,  204, 80, 0, 17, 0.20, 0, 5),  # Suiza
    (17, 209, 87, 0,  1, 0.14, 1, 1),  # Alemania
    (17, 153, 73, 0, 17, 0.55, 0, 9),  # Corea del Sur
    (17, 200, 78, 0, 17, 0.22, 0, 5),  # Polonia
    (17, 183, 73, 0, 17, 0.38, 0, 9),  # Senegal
    (17, 173, 72, 0, 17, 0.42, 0, 9),  # Irán
    (17, 168, 76, 0, 17, 0.45, 0, 9),  # Marruecos
    (17, 177, 71, 0, 17, 0.35, 0, 9),  # Costa Rica
    (17, 188, 71, 0, 17, 0.30, 0, 9),  # Islandia
    (17, 199, 73, 0, 17, 0.25, 0, 9),  # Perú
    (17, 170, 70, 0, 17, 0.45, 0,13),  # Australia
    (17, 162, 74, 0, 17, 0.52, 0, 9),  # Nigeria
    (17, 147, 66, 0, 17, 0.68, 0,13),  # Arabia Saudita
    (17, 165, 76, 0, 17, 0.48, 0, 9),  # Egipto
    (17, 155, 62, 0, 33, 0.72, 0,13),  # Panamá
    (17, 179, 69, 0, 17, 0.40, 0, 9),  # Túnez
    (17, 172, 75, 0, 17, 0.42, 0, 9),  # Serbia
    # ── QATAR 2022 ───────────────────────────────────────────────────────────
    (1,  207, 92, 0,  1, 0.10, 2,  1),  # Argentina
    (2,  206, 88, 0,  1, 0.14, 0,  3),  # Francia
    (3,  198, 84, 0,  2, 0.20, 0,  3),  # Croacia
    (4,  202, 83, 0, 17, 0.38, 1,  1),  # Marruecos
    (5,  209, 88, 0,  5, 0.10, 0,  2),  # Brasil
    (5,  203, 85, 0,  2, 0.22, 0,  3),  # Países Bajos
    (5,  201, 88, 0,  9, 0.12, 1,  1),  # Portugal
    (5,  206, 86, 0,  4, 0.14, 0,  2),  # Inglaterra
    (9,  203, 86, 0,  1, 0.14, 1,  1),  # España
    (9,  184, 78, 0, 17, 0.32, 0,  5),  # Polonia
    (9,  176, 79, 0,  9, 0.35, 0,  2),  # Japón
    (9,  196, 80, 0, 17, 0.25, 1,  1),  # Senegal
    (9,  177, 79, 0,  9, 0.38, 0,  3),  # Corea del Sur
    (9,  172, 74, 0,  9, 0.42, 0,  5),  # Australia
    (9,  194, 79, 0,  9, 0.22, 1,  2),  # Estados Unidos
    (9,  195, 80, 0, 17, 0.20, 0,  5),  # Suiza
    (17, 199, 86, 0,  1, 0.14, 0,  5),  # Alemania
    (17, 208, 87, 0,  5, 0.17, 0,  5),  # Bélgica
    (17, 196, 83, 0,  4, 0.22, 0,  3),  # Uruguay
    (17, 195, 79, 0, 17, 0.25, 1,  2),  # México
    (17, 200, 79, 0, 17, 0.20, 0,  5),  # Dinamarca
    (17, 169, 75, 0, 33, 0.42, 0,  4),  # Canadá
    (17, 157, 73, 0, 17, 0.50, 0,  9),  # Camerún
    (17, 189, 76, 0, 17, 0.30, 0,  9),  # Serbia
    (17, 180, 71, 0, 17, 0.40, 0,  9),  # Túnez
    (17, 166, 74, 0, 17, 0.42, 0,  9),  # Ecuador
    (17, 190, 72, 0, 17, 0.32, 0,  9),  # Irán
    (17, 160, 67, 0, 17, 0.58, 2,  1),  # Qatar
    (17, 159, 68, 0, 17, 0.60, 0,  9),  # Arabia Saudita
    (17, 150, 72, 0, 17, 0.62, 0,  9),  # Ghana
    (17, 179, 70, 0, 17, 0.50, 0,  9),  # Costa Rica
    (17, 191, 75, 0,  3, 0.28, 0,  9),  # Gales
]

HIST_COLS = ["posicion_final","ranking_score","rating_approx","primera_vez",
             "pos_prev_mundial","elim_pct","copa_titulos_ult2","copa_mejor_pos_ult2"]

df_hist = pd.DataFrame(HIST, columns=HIST_COLS)

print(f"✅ Datos históricos: {len(df_hist)} muestras de entrenamiento")
print(f"   Mundiales: 2010 ({32}), 2014 ({32}), 2018 ({32}), 2022 ({32})")
df_hist.describe().round(2)


✅ Datos históricos: 128 muestras de entrenamiento
   Mundiales: 2010 (32), 2014 (32), 2018 (32), 2022 (32)


,posicion_final,ranking_score,rating_approx,primera_vez,pos_prev_mundial,elim_pct,copa_titulos_ult2,copa_mejor_pos_ult2
count,128.00,128.00,128.00,128.0,128.00,128.00,128.00,128.00
mean,11.69,185.84,78.10,0.0,12.62,0.32,0.24,6.11
std,5.70,19.14,7.26,0.0,8.21,0.17,0.48,3.85
min,1.00,105.00,55.00,0.0,1.00,0.10,0.00,1.00
25%,8.00,175.50,73.00,0.0,4.75,0.17,0.00,3.00
50%,13.00,190.50,78.00,0.0,17.00,0.30,0.00,5.00
75%,17.00,201.00,84.25,0.0,17.00,0.42,0.00,9.00
max,17.00,209.00,92.00,0.0,33.00,0.90,2.00,13.00


---
## 4. Feature Engineering

### Variables finales del modelo (v8)

| Variable | Escala | Descripción | Gain esperado |
|----------|--------|-------------|---------------|
| `wc_score` | 0–3.5 | WC2022×2.0 + WC2018×1.5 | **Mayor** |
| `elim_norm` | 0–1 | Fuerza Final eliminatorias (normalizada) | 2° |
| `copa_norm` | 0–1 | Poder Final copas intercontinentales (normalizada) | 3° |
| `ranking_score` | 1–209 | 210 − ranking FIFA | 4° |
| `tm_cat` | 0–3 | Categoría Transfermarkt | 5° |
| `copa_score` | 0–0.96+ | Copa continental × w_conf (últimos 2 torneos) | 6° |
| `rating_eafc26` | ÷10 | EA FC 26 (señal reducida → menor gain) | **Menor** |
| `primera_vez` | 0/1 | Debut absoluto → gain ≈ 0 | ~0 |

### Por qué `elim_norm` y `copa_norm` en lugar de `torneo_score`

`torneo_score = CWC×2.8 + FIC×2.2 + Elim×1.8` tenía gain bajo (0.047) porque:
1. Mezclaba señales de distinta naturaleza en un solo número
2. Las ponderaciones (2.8/2.2/1.8) eran arbitrarias
3. Los valores del CWC y FIC (0–1) son mucho más pequeños que los de eliminatorias (55–170)

Ahora cada variable viene **directamente de los CSVs** con sus propias fórmulas:
- `elim_fuerza`: Fuerza Base (suma de rankings de rivales directos) − Penalidad por posición
- `copa_poder`: Fuerza Base + Ajuste por confederación (UEFA=+15, CONMEBOL=+7, AFC=+2, CAF=+1, CONCACAF=+2, OFC=−10)

Ambas se normalizan a [0,1] para comparabilidad con las demás features.


In [4]:
FEATURES = [
    "wc_score",       # WC2022×2.0 + WC2018×1.5          → gain máximo
    "elim_norm",      # Fuerza Final eliminatorias norm.  → 2°
    "copa_norm",      # Poder Final copas intcont. norm.  → 3°
    "ranking_score",  # FIFA Score                        → 4°
    "tm_cat",         # Categoría Transfermarkt (0-3)     → 5°
    "copa_score",     # Copa continental × w_conf         → 6°
    "rating_eafc26",  # EA FC 26 ÷ 10                     → menor gain
    "primera_vez",    # Debut absoluto                    → ~0
]

ef_min, ef_max = 55.0, 170.0
cp_min, cp_max = 90.0, 185.0

def preparar_features_v8(df):
    X = df[FEATURES].copy().astype(float)
    X["rating_eafc26"] = X["rating_eafc26"] / 10.0   # reducir señal
    return X.values

# ── Datos históricos WC2022 + WC2018 ─────────────────────────────────────────
# Aproximamos elim_norm y copa_norm históricamente usando:
#   - Equipos CONMEBOL: elim_fuerza ≈ 155-165 → norm ≈ 0.85-0.95
#   - UEFA top: elim_fuerza ≈ 140-155 → norm ≈ 0.73-0.87
#   - CAF/AFC: elim_fuerza ≈ 110-115 → norm ≈ 0.48-0.52
#   - copa_poder: basado en rendimiento histórico ×1.0

HIST_2022 = [
    # wc_sc, elim_n, copa_n, rs,  tm, copa_sc, eafc26, pv, pos
    ((1-(1-1)/47)*2.0,  0.91, 0.90, 207, 2, 0.83, 92, 0,  1),  # Argentina
    ((1-(2-1)/47)*2.0,  0.79, 0.97, 206, 3, 0.29, 88, 0,  2),  # Francia
    ((1-(3-1)/47)*2.0,  0.64, 0.81, 198, 1, 0.28, 84, 0,  3),  # Croacia
    ((1-(4-1)/47)*2.0,  0.52, 0.73, 202, 1, 0.51, 83, 0,  4),  # Marruecos
    ((1-(5-1)/47)*2.0,  0.87, 0.86, 209, 3, 0.25, 88, 0,  5),  # Brasil
    ((1-(5-1)/47)*2.0,  0.55, 0.97, 203, 3, 0.29, 85, 0,  5),  # Países Bajos
    ((1-(5-1)/47)*2.0,  0.83, 0.89, 201, 3, 0.46, 88, 0,  5),  # Portugal
    ((1-(5-1)/47)*2.0,  0.56, 0.89, 206, 3, 0.92, 86, 0,  5),  # Inglaterra
    ((1-(9-1)/47)*2.0,  0.87, 1.00, 203, 3, 0.92, 86, 0,  9),  # España
    ((1-(9-1)/47)*2.0,  0.55, 0.79, 184, 1, 0.25, 78, 0,  9),  # Polonia
    ((1-(9-1)/47)*2.0,  0.50, 0.80, 176, 1, 0.37, 79, 0,  9),  # Japón
    ((1-(9-1)/47)*2.0,  0.52, 0.75, 196, 1, 0.46, 80, 0,  9),  # Senegal
    ((1-(9-1)/47)*2.0,  0.50, 0.82, 177, 1, 0.28, 79, 0,  9),  # Corea del Sur
    ((1-(9-1)/47)*2.0,  0.50, 0.78, 172, 1, 0.27, 74, 0,  9),  # Australia
    ((1-(9-1)/47)*2.0,  0.61, 0.70, 194, 2, 0.44, 79, 0,  9),  # EEUU
    ((1-(9-1)/47)*2.0,  0.83, 0.84, 195, 1, 0.46, 80, 0,  9),  # Suiza
    ((1-(17-1)/47)*2.0, 0.80, 0.87, 199, 3, 0.46, 86, 0, 17),  # Alemania
    ((1-(17-1)/47)*2.0, 0.55, 0.88, 208, 2, 0.46, 87, 0, 17),  # Bélgica
    ((1-(17-1)/47)*2.0, 0.92, 0.87, 196, 1, 0.83, 83, 0, 17),  # Uruguay
    ((1-(17-1)/47)*2.0, 0.61, 0.86, 195, 1, 0.44, 79, 0, 17),  # México
    ((1-(17-1)/47)*2.0, 0.78, 0.80, 200, 1, 0.46, 79, 0, 17),  # Dinamarca
    ((1-(17-1)/47)*2.0, 0.61, 0.89, 169, 1, 0.44, 75, 0, 17),  # Canadá
    ((1-(17-1)/47)*2.0, 0.48, 0.75, 157, 0, 0.48, 73, 0, 17),  # Camerún
    ((1-(17-1)/47)*2.0, 0.62, 0.78, 189, 1, 0.34, 76, 0, 17),  # Serbia
    ((1-(17-1)/47)*2.0, 0.52, 0.65, 180, 0, 0.39, 71, 0, 17),  # Túnez
    ((1-(17-1)/47)*2.0, 0.86, 0.83, 166, 1, 0.25, 74, 0, 17),  # Ecuador
    ((1-(17-1)/47)*2.0, 0.50, 0.84, 190, 0, 0.30, 72, 0, 17),  # Irán
    ((1-(17-1)/47)*2.0, 0.46, 0.86, 160, 0, 0.55, 67, 0, 17),  # Qatar
    ((1-(17-1)/47)*2.0, 0.45, 0.66, 159, 0, 0.27, 68, 0, 17),  # Arabia Saudí
    ((1-(17-1)/47)*2.0, 0.48, 0.10, 150, 0, 0.48, 72, 0, 17),  # Ghana
    ((1-(17-1)/47)*2.0, 0.48, 0.77, 179, 0, 0.34, 70, 0, 17),  # Costa Rica
    ((1-(17-1)/47)*2.0, 0.60, 0.70, 191, 1, 0.32, 75, 0, 17),  # Gales
]

HIST_2018 = [
    ((1-(1-1)/31)*1.5,  0.79, 0.94, 203, 3, 0.29, 87, 0,  1),  # Francia
    ((1-(2-1)/31)*1.5,  0.64, 0.78, 190, 1, 0.28, 84, 0,  2),  # Croacia
    ((1-(3-1)/31)*1.5,  0.55, 0.87, 207, 2, 0.46, 87, 0,  3),  # Bélgica
    ((1-(4-1)/31)*1.5,  0.56, 0.89, 198, 3, 0.46, 84, 0,  4),  # Inglaterra
    ((1-(5-1)/31)*1.5,  0.92, 0.87, 196, 1, 0.83, 84, 0,  5),  # Uruguay
    ((1-(5-1)/31)*1.5,  0.87, 0.86, 208, 3, 0.25, 88, 0,  5),  # Brasil
    ((1-(5-1)/31)*1.5,  0.78, 0.73, 185, 1, 0.42, 76, 0,  5),  # Suecia
    ((1-(5-1)/31)*1.5,  0.55, 0.55, 140, 1, 0.22, 71, 0,  5),  # Rusia
    ((1-(9-1)/31)*1.5,  0.91, 0.87, 205, 2, 0.83, 90, 0,  9),  # Argentina
    ((1-(9-1)/31)*1.5,  0.83, 0.89, 206, 3, 0.46, 87, 0,  9),  # Portugal
    ((1-(9-1)/31)*1.5,  0.87, 1.00, 201, 3, 0.46, 87, 0,  9),  # España
    ((1-(9-1)/31)*1.5,  0.78, 0.80, 191, 1, 0.42, 77, 0,  9),  # Dinamarca
    ((1-(9-1)/31)*1.5,  0.50, 0.80, 149, 1, 0.37, 74, 0,  9),  # Japón
    ((1-(9-1)/31)*1.5,  0.61, 0.86, 195, 1, 0.44, 80, 0,  9),  # México
    ((1-(9-1)/31)*1.5,  0.90, 0.89, 194, 1, 0.25, 82, 0,  9),  # Colombia
    ((1-(9-1)/31)*1.5,  0.83, 0.84, 204, 1, 0.46, 80, 0,  9),  # Suiza
    ((1-(17-1)/31)*1.5, 0.80, 0.87, 209, 3, 0.46, 87, 0, 17),  # Alemania
    ((1-(17-1)/31)*1.5, 0.50, 0.82, 153, 1, 0.28, 73, 0, 17),  # Corea del Sur
    ((1-(17-1)/31)*1.5, 0.55, 0.79, 200, 1, 0.46, 78, 0, 17),  # Polonia
    ((1-(17-1)/31)*1.5, 0.52, 0.75, 183, 1, 0.50, 73, 0, 17),  # Senegal
    ((1-(17-1)/31)*1.5, 0.50, 0.84, 173, 0, 0.30, 72, 0, 17),  # Irán
    ((1-(17-1)/31)*1.5, 0.52, 0.73, 168, 1, 0.51, 76, 0, 17),  # Marruecos
    ((1-(17-1)/31)*1.5, 0.48, 0.77, 177, 0, 0.34, 71, 0, 17),  # Costa Rica
    ((1-(17-1)/31)*1.5, 0.60, 0.65, 188, 0, 0.22, 71, 0, 17),  # Islandia
    ((1-(17-1)/31)*1.5, 0.91, 0.87, 199, 1, 0.83, 73, 0, 17),  # Perú
    ((1-(17-1)/31)*1.5, 0.50, 0.78, 170, 1, 0.27, 70, 0, 17),  # Australia
    ((1-(17-1)/31)*1.5, 0.48, 0.75, 162, 1, 0.48, 74, 0, 17),  # Nigeria
    ((1-(17-1)/31)*1.5, 0.45, 0.66, 147, 0, 0.27, 66, 0, 17),  # Arabia Saudí
    ((1-(17-1)/31)*1.5, 0.48, 0.75, 165, 1, 0.37, 76, 0, 17),  # Egipto
    ((1-(17-1)/31)*1.5, 0.48, 0.77, 155, 0, 0.44, 62, 0, 17),  # Panamá
    ((1-(17-1)/31)*1.5, 0.52, 0.65, 179, 0, 0.39, 69, 0, 17),  # Túnez
    ((1-(17-1)/31)*1.5, 0.62, 0.78, 172, 1, 0.34, 75, 0, 17),  # Serbia
]

HIST_COLS = ["wc_score","elim_norm","copa_norm","ranking_score","tm_cat",
             "copa_score","rating_eafc26","primera_vez","posicion_final"]

df_h22 = pd.DataFrame([(*r[:8], r[8]) for r in HIST_2022], columns=HIST_COLS)
df_h18 = pd.DataFrame([(*r[:8], r[8]) for r in HIST_2018], columns=HIST_COLS)
df_hist_model = pd.concat([df_h22, df_h18], ignore_index=True)
sample_weights = np.array([2.0]*32 + [1.5]*32)

X_train_raw = preparar_features_v8(df_hist_model)
y_train     = df_hist_model["posicion_final"].values
X_2026_raw  = preparar_features_v8(df_2026)

scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_2026  = scaler.transform(X_2026_raw)

print("✅ Features v8 preparados — elim_norm y copa_norm desde CSV reales")
print(f"   Features: {FEATURES}")
print(f"   X_train : {X_train.shape} | X_2026: {X_2026.shape}")
print()
print("Verificación elim_norm y copa_norm — Top 5 equipos 2026:")
df_check = df_2026[["equipo","elim_fuerza","elim_norm","copa_poder","copa_norm","wc_score"]].sort_values("wc_score",ascending=False).head(5)
print(df_check.to_string(index=False))


✅ Features v8 preparados — elim_norm y copa_norm desde CSV reales
   Features: ['wc_score', 'elim_norm', 'copa_norm', 'ranking_score', 'tm_cat', 'copa_score', 'rating_eafc26', 'primera_vez']
   X_train : (64, 8) | X_2026: (48, 8)

Verificación elim_norm y copa_norm — Top 5 equipos 2026:
    equipo  elim_fuerza  elim_norm  copa_poder  copa_norm  wc_score
   Francia       146.25   0.793478      181.45   0.962632    3.4574
   Croacia       129.20   0.645217      168.15   0.822632    3.3665
Inglaterra       119.80   0.563478      175.00   0.894737    3.0144
 Argentina       165.22   0.958435      177.00   0.915789    2.7258
 Marruecos       115.00   0.521739      158.45   0.720526    2.5981



---
## 5. Entrenamiento XGBoost

### Hiperparámetros

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `max_depth` | **5** | Máximo permitido — captura relaciones no lineales sin sobreajuste |
| `n_estimators` | 300 | Suficientes árboles para convergencia |
| `learning_rate` | 0.05 | Conservador, reduce sobreajuste |
| `subsample` | 0.8 | Muestreo parcial de filas por árbol |
| `colsample_bytree` | 0.8 | Muestreo parcial de features por árbol |
| `min_child_weight` | 3 | Mínimo de muestras por hoja |
| `reg_alpha` | 0.1 | Regularización L1 |
| `reg_lambda` | 1.0 | Regularización L2 |

### Validación
Con solo 128 muestras históricas, usamos **Leave-One-Out Cross-Validation (LOO-CV)**,
que maximiza los datos de entrenamiento en cada iteración.


In [5]:
MODEL_PARAMS = dict(
    max_depth=5,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=8.0,      # Alto para penalizar features de baja señal (eafc26/10)
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("⏳ Ejecutando LOO-CV (64 iteraciones, sample_weight)...")
loo = LeaveOneOut()
errores_loo = []

for train_idx, test_idx in loo.split(X_train):
    m = XGBRegressor(**MODEL_PARAMS)
    m.fit(X_train[train_idx], y_train[train_idx],
          sample_weight=sample_weights[train_idx])
    pred = m.predict(X_train[test_idx])
    errores_loo.append(abs(pred[0] - y_train[test_idx][0]))

mae_loo = np.mean(errores_loo)
std_loo = np.std(errores_loo)
print(f"✅ LOO-CV completado")
print(f"   MAE promedio : {mae_loo:.2f} posiciones")
print(f"   Std          : {std_loo:.2f}")
print(f"   Este MAE se usa como σ del ruido en el fixture de grupos")

model = XGBRegressor(**MODEL_PARAMS)
model.fit(X_train, y_train, sample_weight=sample_weights)
print(f"\n✅ Modelo entrenado | reg_lambda=8.0 | sample_weight WC2022×2.0, WC2018×1.5")


⏳ Ejecutando LOO-CV (64 iteraciones, sample_weight)...
✅ LOO-CV completado
   MAE promedio : 1.80 posiciones
   Std          : 2.00
   Este MAE se usa como σ del ruido en el fixture de grupos

✅ Modelo entrenado | reg_lambda=8.0 | sample_weight WC2022×2.0, WC2018×1.5


---
## 6. Predicción 2026

El modelo genera un **score continuo** por equipo. Ordenamos ascendentemente
y asignamos posiciones 1–48 con la siguiente lógica corregida:

| Posición | Fase | Equipos en fase | Eliminados en fase |
|----------|------|----------------|-------------------|
| 1 | 🏆 Campeón | 1 | — |
| 2 | 🥈 Subcampeón | 1 | — |
| 3 | 🥉 3er lugar | 1 | — |
| 4 | 4to lugar | 1 | — |
| 5–8 | Cuartos de final | 4 llegan, **4 eliminados** | 4 |
| 9–16 | Octavos de final | 8 llegan, **8 eliminados** | 8 |
| 17–32 | Dieciseisavos | 16 llegan, **16 eliminados** | 16 |
| 33–48 | Fase de grupos | 16 eliminados | 16 |
| **Total** | | **48** | **48 ✅** |


In [6]:
# ── Predicción ────────────────────────────────────────────────────────────────
df_2026["score"] = model.predict(X_2026)
df_2026 = df_2026.sort_values("score", ascending=True).reset_index(drop=True)
df_2026["posicion"] = df_2026.index + 1

# ── Mapeo posición → fase ─────────────────────────────────────────────────────
def pos_a_fase(pos):
    if pos == 1:    return "Campeón",       0
    if pos == 2:    return "Subcampeón",    1
    if pos == 3:    return "3er lugar",     2
    if pos == 4:    return "4to lugar",     3
    if pos <= 8:    return "Cuartos",       4
    if pos <= 16:   return "Octavos",       5
    if pos <= 32:   return "Dieciseisavos", 6
    return                 "Grupos",        7

FASE_LABEL = {0:"🏆 Campeón", 1:"🥈 Subcampeón", 2:"🥉 3er lugar",
              3:"4to lugar",  4:"Cuartos de final", 5:"Octavos de final",
              6:"Dieciseisavos", 7:"Fase de grupos"}
FASE_COLOR = {0:"#FFD700", 1:"#C0C0C0", 2:"#CD7F32", 3:"#4CAF50",
              4:"#2196F3", 5:"#7B1FA2", 6:"#9C27B0", 7:"#607D8B"}

df_2026["fase_idx"]  = df_2026["posicion"].apply(lambda p: pos_a_fase(p)[1])
df_2026["fase"]      = df_2026["fase_idx"].map(FASE_LABEL)
df_2026["fase_color"]= df_2026["fase_idx"].map(FASE_COLOR)

# ── Restricción XGBoost → fixture (±1 ronda) ─────────────────────────────────
# Se guarda la fase predicha como "banda" para el simulador de partidos.
# En sim_elim(), si el resultado implicaría que un equipo avance MÁS de fase_idx+1
# o salga ANTES de fase_idx-1, se ajusta la probabilidad para corregirlo.
# Esto se implementa en la función constrained_sim_elim() de la sección 14.

fase_predicha = dict(zip(df_2026["equipo"], df_2026["fase_idx"]))
score_map     = dict(zip(df_2026["equipo"], df_2026["score"]))

print("═" * 64)
print("  PREDICCIÓN MUNDIAL 2026 — XGBoost v4 (max_depth=5)")
print("  Variables: FIFA Score · EAFC26 · CWC×2.8 · FIC×2.2 · Elim×1.8")
print("═" * 64)
print(f"{'POS':>4}  {'EQUIPO':<22} {'GRP':>4}  {'FASE'}")
print("─" * 64)
for _, r in df_2026.iterrows():
    print(f"{int(r['posicion']):>4}  {r['equipo']:<22} {'Gr.'+r['grupo']:>4}  {r['fase']}")
print("═" * 64)
print()
conteo = df_2026.groupby("fase")["equipo"].count()
orden  = ["🏆 Campeón","🥈 Subcampeón","🥉 3er lugar","4to lugar",
          "Cuartos de final","Octavos de final","Dieciseisavos","Fase de grupos"]
print("Verificación de conteo por fase:")
total = 0
for f in orden:
    n = conteo.get(f, 0)
    total += n
    if n: print(f"  {f:<25} {n:>2} equipos")
print(f"  {'TOTAL':<25} {total:>2} ✅" if total==48 else f"  TOTAL: {total} ⚠️")
print(f"\n📊 MAE validación LOO: {mae_loo:.2f} posiciones (±{std_loo:.2f})")


════════════════════════════════════════════════════════════════
  PREDICCIÓN MUNDIAL 2026 — XGBoost v4 (max_depth=5)
  Variables: FIFA Score · EAFC26 · CWC×2.8 · FIC×2.2 · Elim×1.8
════════════════════════════════════════════════════════════════
 POS  EQUIPO                  GRP  FASE
────────────────────────────────────────────────────────────────
   1  Inglaterra             Gr.L  🏆 Campeón
   2  Portugal               Gr.K  🥈 Subcampeón
   3  Países Bajos           Gr.F  🥉 3er lugar
   4  Francia                Gr.I  4to lugar
   5  Brasil                 Gr.C  Cuartos de final
   6  Suecia                 Gr.F  Cuartos de final
   7  Argentina              Gr.J  Cuartos de final
   8  España                 Gr.H  Cuartos de final
   9  Alemania               Gr.E  Octavos de final
  10  Ecuador                Gr.E  Octavos de final
  11  Croacia                Gr.L  Octavos de final
  12  Marruecos              Gr.C  Octavos de final
  13  Bélgica                Gr.G  Octavos de f


---
## 7. Dashboard — Ranking final

Visualización interactiva del ranking completo de los 48 equipos.
Hover sobre cada barra para ver detalles del equipo.


In [7]:
# ── Dashboard 1: Ranking horizontal bar chart ─────────────────────────────────
TOP = df_2026.copy()

fig1 = go.Figure()

for phase, group in TOP.groupby("fase", sort=False):
    color = group["fase_color"].iloc[0]
    fig1.add_trace(go.Bar(
        y=group["equipo"],
        x=group["ranking_score"],
        orientation="h",
        name=phase.replace("🏆 ","").replace("🥈 ","").replace("🥉 ",""),
        marker=dict(
            color=color,
            opacity=0.88,
            line=dict(color="rgba(255,255,255,0.15)", width=0.5)
        ),
        customdata=np.stack([
            group["posicion"],
            group["grupo"],
            group["rating_eafc26"],
            group["fase"],
            group["ranking_fifa"],
        ], axis=-1),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Posición: %{customdata[0]}<br>"
            "Fase: %{customdata[3]}<br>"
            "Grupo: %{customdata[1]}<br>"
            "FIFA Rank: %{customdata[4]}<br>"
            "EAFC26: %{customdata[2]}<br>"
            "FIFA Score: %{x}<extra></extra>"
        ),
    ))

fig1.update_layout(
    title=dict(
        text="<b>Mundial 2026 — Predicción XGBoost</b><br>"
             "<sup>Ranking de los 48 clasificados · Barra = FIFA Score (mayor es mejor)</sup>",
        font=dict(size=18, color="#1a1a2e"),
        x=0.01,
    ),
    barmode="stack",
    plot_bgcolor="#F7F9FC",
    paper_bgcolor="white",
    height=1300,
    yaxis=dict(
        autorange="reversed",
        tickfont=dict(size=11),
        gridcolor="rgba(0,0,0,0.05)",
    ),
    xaxis=dict(
        title="FIFA Score (210 − Ranking FIFA)",
        gridcolor="rgba(0,0,0,0.08)",
    ),
    legend=dict(
        title="Fase",
        orientation="h",
        yanchor="bottom", y=1.01,
        xanchor="right",  x=1,
        font=dict(size=11),
    ),
    margin=dict(l=150, r=30, t=100, b=50),
    font=dict(family="Georgia, serif"),
)

fig1.show()


---
## 8. Dashboard — Feature Importance

Importancia de cada variable según el criterio **gain** de XGBoost.

Las variables nuevas `torneo_score` y `copa_score` incorporan:
- `torneo_score`: **CWC×2.8** (Mayor peso — representa nivel de clubes del país) + **FIC×2.2** + **Eliminatorias×1.8**
- `copa_score`: rendimiento en copa continental **ajustado por dificultad**
  (UEFA=0.96 → OFC=0.11), evitando que ganar la OFC valga lo mismo que ganar la Eurocopa


In [8]:
# ── Dashboard 2: Feature importance ──────────────────────────────────────────
imp_vals = model.feature_importances_
imp_df   = pd.DataFrame({"feature": FEATURES, "importance": imp_vals})
imp_df   = imp_df.sort_values("importance", ascending=True)

FEAT_LABELS = {
    "ranking_score": "FIFA Score (210 − Ranking)",
    "rating_eafc26": "EAFC26 Squad Rating",
    "primera_vez":   "Debut absoluto",
    "tm_value_log":  "Valor Transfermarkt log(€M)",
    "elim_norm":     "Fuerza Final Eliminatorias (CSV norm.)",
    "copa_norm":     "Poder Final Copas Intercontinentales (CSV norm.)",
    "copa_score":    "Copa Cont. × Dificultad Confederación",
    "wc_score":      "WC Score (2022×2.0 + 2018×1.5)",
}

labels = [FEAT_LABELS.get(f, f) for f in imp_df["feature"]]
vals   = imp_df["importance"].values
palette = ["#B0BEC5","#90A4AE","#78909C","#546E7A","#1E88E5","#1565C0","#0d1b2a"]

fig2 = go.Figure(go.Bar(
    x=vals, y=labels, orientation="h",
    marker=dict(color=palette[:len(vals)], line=dict(color="white", width=0.5)),
    text=[f"{v:.3f}" for v in vals], textposition="outside",
    hovertemplate="<b>%{y}</b><br>Importancia: %{x:.4f}<extra></extra>",
))
fig2.update_layout(
    title=dict(
        text="<b>Importancia de Variables — XGBoost v5 (gain)</b><br>"
             "<sup>tm_value_log (Transfermarkt) · wc_score (WC2022×2.0+WC2018×1.5) · copa_score (×w_conf)</sup>",
        font=dict(size=17, color="#1a1a2e"), x=0.01,
    ),
    plot_bgcolor="#F7F9FC", paper_bgcolor="white", height=450,
    xaxis=dict(title="Feature Importance (gain)", gridcolor="rgba(0,0,0,0.08)"),
    yaxis=dict(tickfont=dict(size=11)),
    margin=dict(l=260, r=80, t=90, b=50),
    font=dict(family="Georgia, serif"),
)
fig2.show()



---
## 9. Dashboard — Grupos & Clasificados

Vista de los 12 grupos con el score XGBoost de cada equipo.
Los 2 primeros de cada grupo (y los 8 mejores terceros) clasifican a dieciseisavos.


In [9]:
# ── Dashboard 3: grupos — scatter por grupo ───────────────────────────────────
grupos = sorted(df_2026["grupo"].unique())
fig3 = make_subplots(
    rows=3, cols=4,
    subplot_titles=[f"<b>Grupo {g}</b>" for g in grupos],
    vertical_spacing=0.12,
    horizontal_spacing=0.06,
)

for idx, g in enumerate(grupos):
    r, c = divmod(idx, 4)
    gdf  = df_2026[df_2026["grupo"] == g].sort_values("score")
    
    for i, (_, row) in enumerate(gdf.iterrows()):
        marker_color = "#FFD700" if i == 0 else ("#90CAF9" if i == 1 else "#CFD8DC")
        fig3.add_trace(
            go.Bar(
                x=[row["equipo"]],
                y=[row["score"]],
                marker_color=marker_color,
                marker_line_color="rgba(0,0,0,0.2)",
                marker_line_width=0.8,
                name=row["equipo"],
                showlegend=False,
                hovertemplate=(
                    f"<b>{row['equipo']}</b><br>"
                    f"Score: {row['score']:.2f}<br>"
                    f"Posición predicha: {int(row['posicion'])}<br>"
                    f"Fase: {row['fase']}<extra></extra>"
                ),
            ),
            row=r+1, col=c+1,
        )

fig3.update_layout(
    title=dict(
        text="<b>Scores XGBoost por Grupo</b><br>"
             "<sup>🥇 Oro = 1°  🔵 Azul = 2°  ⬜ Gris = eliminado</sup>",
        font=dict(size=17, color="#1a1a2e"),
        x=0.01,
    ),
    plot_bgcolor="#F7F9FC",
    paper_bgcolor="white",
    height=800,
    font=dict(family="Georgia, serif", size=10),
    margin=dict(l=40, r=40, t=100, b=40),
)

for ax in fig3.layout:
    if ax.startswith("yaxis"):
        fig3.layout[ax].update(showticklabels=False, gridcolor="rgba(0,0,0,0.06)")
    if ax.startswith("xaxis"):
        fig3.layout[ax].update(tickfont=dict(size=8))

fig3.show()



---
## 10. Dashboard — Radar por equipo

Perfil multidimensional de las variables normalizadas para los **Top 16** clasificados.
Selecciona el equipo usando la leyenda interactiva.


In [10]:
# ── Dashboard 4: Radar chart Top 16 ──────────────────────────────────────────
top16 = df_2026[df_2026["posicion"] <= 16].copy().reset_index(drop=True)

tm_max = df_2026["tm_cat"].max()

top16["radar_ranking"]  = top16["ranking_score"] / 209
top16["radar_rating"]   = (top16["rating_eafc26"] - 50) / 42
top16["radar_tm"]       = top16["tm_cat"] / tm_max
top16["radar_elim"]     = top16["elim_norm"]
top16["radar_copa_intl"] = top16["copa_norm"]
top16["radar_copa"]     = top16["copa_score"]   / df_2026["copa_score"].max().clip(0.01)
top16["radar_wc"]       = top16["wc_score"]     / df_2026["wc_score"].max().clip(0.01)

radar_cols   = ["radar_ranking","radar_rating","radar_tm",
                "radar_elim","radar_copa_intl","radar_copa","radar_wc"]
radar_labels = ["FIFA Score","EAFC26 Rating","Transfermarkt (€M)",
                "Elim.\nFuerza","Copas\nIntcont.","Copa Cont.\n(×conf.)","WC Score\n(2022+2018)"]

PALETTE = ["#FFD700","#C0C0C0","#CD7F32","#4CAF50","#2196F3","#F44336",
           "#9C27B0","#00BCD4","#FF5722","#8BC34A","#3F51B5","#E91E63",
           "#009688","#FF9800","#795548","#607D8B"]

fig4 = go.Figure()
for i, (_, row) in enumerate(top16.iterrows()):
    vals   = [row[c] for c in radar_cols] + [row[radar_cols[0]]]
    labels = radar_labels + [radar_labels[0]]
    r,g,b  = int(PALETTE[i][1:3],16), int(PALETTE[i][3:5],16), int(PALETTE[i][5:7],16)
    fig4.add_trace(go.Scatterpolar(
        r=vals, theta=labels, fill="toself",
        name=f"{int(row['posicion'])}. {row['equipo']}",
        line=dict(color=PALETTE[i], width=2),
        fillcolor=f"rgba({r},{g},{b},0.07)",
        hovertemplate=f"<b>{row['equipo']}</b><br>%{{theta}}: %{{r:.2f}}<extra></extra>",
        visible="legendonly" if i > 3 else True,
    ))

fig4.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0,1], tickfont=dict(size=9),
                        gridcolor="rgba(0,0,0,0.10)"),
        angularaxis=dict(tickfont=dict(size=10)),
        bgcolor="#F7F9FC",
    ),
    title=dict(
        text="<b>Perfil Multidimensional — Top 16</b><br>"
             "<sup>6 dimensiones · Mayor área = mejor perfil global · Doble clic para aislar</sup>",
        font=dict(size=17, color="#1a1a2e"), x=0.01,
    ),
    legend=dict(font=dict(size=10), itemclick="toggle", itemdoubleclick="toggleothers"),
    paper_bgcolor="white", height=640,
    font=dict(family="Georgia, serif"),
    margin=dict(l=60, r=200, t=100, b=50),
)
fig4.show()



---
## 11. Dashboard — Bubble Chart: Score vs Ranking FIFA

Cada burbuja es un equipo. El **eje X** es el FIFA Score, el **eje Y** es el EAFC26 Rating,
el **tamaño** refleja los títulos continentales recientes y el **color** la fase predicha.


In [11]:
# ── Dashboard 5: Bubble — FIFA Score vs Transfermarkt value ──────────────────
fig5 = go.Figure()

fase_order = ["🏆 Campeón","🥈 Subcampeón","🥉 3er lugar","4to lugar",
              "Cuartos de final","Octavos de final","Dieciseisavos","Fase de grupos"]
fase_colors_map = {
    "🏆 Campeón":"#FFD700","🥈 Subcampeón":"#C0C0C0","🥉 3er lugar":"#CD7F32",
    "4to lugar":"#4CAF50","Cuartos de final":"#2196F3","Octavos de final":"#7B1FA2",
    "Dieciseisavos":"#9C27B0","Fase de grupos":"#607D8B",
}

# Tamaño burbuja = wc_score (experiencia mundialista reciente)
wc_min = df_2026["wc_score"].min()
wc_max = df_2026["wc_score"].max()
df_2026["bubble_size"] = 10 + 30 * (df_2026["wc_score"] - wc_min) / (wc_max - wc_min + 1e-9)

for fase_label in fase_order:
    sub = df_2026[df_2026["fase"] == fase_label]
    if sub.empty: continue
    fig5.add_trace(go.Scatter(
        x=sub["ranking_score"],
        y=sub["tm_value_m"],
        mode="markers+text",
        name=fase_label.replace("🏆 ","").replace("🥈 ","").replace("🥉 ",""),
        marker=dict(size=sub["bubble_size"], color=fase_colors_map[fase_label],
                    opacity=0.82, line=dict(color="white", width=1.2), sizemode="diameter"),
        text=sub["equipo"].str[:3].str.upper(),
        textposition="middle center",
        textfont=dict(size=8, color="white", family="Georgia, serif"),
        customdata=np.stack([sub["equipo"], sub["posicion"], sub["grupo"],
                             sub["tm_value_m"], sub["wc_score"].round(2),
                             sub["elim_norm"].round(3)], axis=-1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Posición XGBoost: %{customdata[1]}<br>"
            "Grupo: %{customdata[2]}<br>"
            "FIFA Score: %{x}<br>"
            "Transfermarkt: €%{customdata[3]}M<br>"
            "WC Score: %{customdata[4]}<br>"
            "Elim. Fuerza norm.: %{customdata[5]}<extra></extra>"
        ),
    ))

fig5.update_layout(
    title=dict(
        text="<b>FIFA Score vs Valor Transfermarkt — 48 Selecciones</b><br>"
             "<sup>X=FIFA Score · Y=Valor Transfermarkt (€M) · Tamaño=WC Score (2022×2.0+2018×1.5)</sup>",
        font=dict(size=17, color="#1a1a2e"), x=0.01,
    ),
    xaxis=dict(title="FIFA Score (210 − Ranking)", gridcolor="rgba(0,0,0,0.08)", zeroline=False),
    yaxis=dict(title="Valor de mercado Transfermarkt (€M)", gridcolor="rgba(0,0,0,0.08)",
               zeroline=False, type="log", title_text="Transfermarkt €M (escala log)"),
    plot_bgcolor="#F7F9FC", paper_bgcolor="white", height=640,
    legend=dict(title="Fase predicha", font=dict(size=11),
                itemclick="toggle", itemdoubleclick="toggleothers"),
    font=dict(family="Georgia, serif"),
    margin=dict(l=80, r=200, t=110, b=60),
)
fig5.show()



---
## 12. Dashboard — Heatmap de variables por grupo

Vista de calor de todas las variables normalizadas para cada equipo,
agrupadas por grupo del sorteo.


In [12]:
from sklearn.preprocessing import MinMaxScaler as MMS
import plotly.graph_objects as go

# ── CONTROL DE SEGURIDAD ─────────────────────────────────────────────────────
if 'mt_equipos' not in locals() and 'mt_equipos' not in globals():
    mt_equipos = []

col_grupo = "grupo" if "grupo" in df_2026.columns else "Grupo"
col_equipo = "equipo" if "equipo" in df_2026.columns else "Equipo"

if "score_adj" in df_2026.columns:
    col_score = "score_adj"
elif "score" in df_2026.columns:
    col_score = "score"
else:
    columnas_score = [c for c in df_2026.columns if "score" in c.lower()]
    col_score = columnas_score[0] if columnas_score else df_2026.columns[-1]

# ── PROCESAMIENTO DE DATOS ───────────────────────────────────────────────────
df_heat = df_2026.sort_values([col_grupo, col_score]).copy().reset_index(drop=True)
df_heat["pos_en_grupo"] = df_heat.groupby(col_grupo)[col_score].rank(method="first").astype(int)

feat_heat  = ["wc_score", "elim_norm", "copa_norm",
              "ranking_score", "tm_cat", "copa_score",
              "rating_eafc26", "primera_vez"]
feat_names = ["WC Score (2022+18)", "Elim. Fuerza", "Copas Intcont.",
              "FIFA Score", "TM Cat", "Copa Cont.", "EAFC26", "Debut"]

tmp = df_heat[feat_heat].copy().astype(float)
tmp["primera_vez"] = 1.0 - tmp["primera_vez"]

heat_vals = MMS().fit_transform(tmp)

labels_y = []
for _, row in df_heat.iterrows():
    pos = int(row["pos_en_grupo"])
    if pos <= 2:
        mark = "★ "
    elif pos == 3 and row[col_equipo] in mt_equipos:
        mark = "· "
    else:
        mark = "  "
    labels_y.append(f"{mark}Gr.{row[col_grupo]} {row[col_equipo]}")

group_boundaries = []
if len(df_heat) > 0:
    cur_g = df_heat.iloc[0][col_grupo]
    for idx_b, row in df_heat.iterrows():
        if row[col_grupo] != cur_g:
            group_boundaries.append(idx_b - 0.5)
            cur_g = row[col_grupo]

shapes = [
    dict(type="line", x0=-0.5, x1=len(feat_heat)-0.5,
         y0=b, y1=b, line=dict(color="rgba(255,255,255,0.7)", width=1.5))
    for b in group_boundaries
]

# ── GENERACIÓN DEL GRÁFICO ───────────────────────────────────────────────────
fig6 = go.Figure(go.Heatmap(
    z=heat_vals,
    x=feat_names,
    y=labels_y,
    colorscale=[
        [0.00, "#0d1b2a"],
        [0.25, "#1565C0"],
        [0.50, "#42A5F5"],
        [0.75, "#FFD54F"],
        [1.00, "#FFD700"],
    ],
    text=[[f"{v:.2f}" for v in row] for row in heat_vals],
    texttemplate="%{text}",
    textfont=dict(size=8), 
    hoverongaps=False,
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:.3f}<extra></extra>",
    showscale=True,
    # SOLUCIÓN DEL ERROR AQUÍ:
    colorbar=dict(
        title=dict(
            text="Score norm.",
            side="right"
        ),
        tickfont=dict(size=10),
        len=0.55,
    ),
))

fig6.update_layout(
    title=dict(
        text=(
            "<b>Heatmap de Variables — 48 Selecciones por Grupo</b><br>"
            "<sup>Norm. 0–1 · Amarillo=mayor valor · "
            "★=clasificado (1°/2°) · ·=mejor 3° · Líneas blancas=cambio de grupo</sup>"
        ),
        font=dict(size=17, color="#1a1a2e"),
        x=0.01,
    ),
    height=1450,
    plot_bgcolor="#0d1b2a",
    paper_bgcolor="white",
    xaxis=dict(side="top", tickfont=dict(size=10), tickangle=-25), 
    yaxis=dict(tickfont=dict(size=9, family="monospace"), autorange="reversed"), 
    font=dict(family="Georgia, serif"),
    margin=dict(l=240, r=80, t=140, b=40),
    shapes=shapes,
)

fig6.show()

In [13]:
import numpy as np, pandas as pd
from itertools import combinations

np.random.seed(42)

# ── Score map y fase predicha ────────────────────────────────────────────────
score_map      = dict(zip(df_2026["equipo"], df_2026["score"]))
fase_predicha  = dict(zip(df_2026["equipo"], df_2026["fase_idx"]))
debut_equipos  = set(df_2026[df_2026["primera_vez"]==1]["equipo"].tolist())
GRUPOS         = sorted(df_2026["grupo"].unique())




---
## 14. Bracket Eliminatorias — XGBoost Clasificador Binario + Fixture Oficial FIFA 2026

### Modelo: Clasificador Binario `binary:logistic`

Para eliminatorias se usa un **segundo XGBoost** entrenado como clasificador:
- **Input**: ΔX = X_A − X_B (diferencia de features entre equipo A y equipo B)
- **Output**: P(A clasifica) ∈ [0, 1]
- Si P > 0.5 → clasifica A; si P < 0.5 → clasifica B
- **Sin marcadores** — solo se determina quién avanza y con qué probabilidad

### Fixture Oficial FIFA 2026 — Dieciseisavos

| Llave | Fecha | Local | Visitante |
|-------|-------|-------|-----------|
| L01 | 30 jun 16:00 | 1° Grupo E | mejor 3° de A/B/C/D/F |
| L02 | 30 jun 16:00 | 1° Grupo I | mejor 3° de C/D/F/G/H |
| L03 | 28 jun 14:00 | 2° Grupo A | 2° Grupo B |
| L04 | 29 jun 20:00 | 1° Grupo F | 2° Grupo C |
| L05 | 2 jul 18:00 | 2° Grupo K | 2° Grupo L |
| L06 | 2 jul 14:00 | 1° Grupo H | 2° Grupo J |
| L07 | 1 jul 19:00 | 1° Grupo D | mejor 3° de B/E/F/I/J |
| L08 | 1 jul 15:00 | 1° Grupo G | mejor 3° de A/E/H/I/J |
| L09 | 29 jun 12:00 | 1° Grupo C | 2° Grupo F |
| L10 | 30 jun 12:00 | 2° Grupo E | 2° Grupo I |
| L11 | 30 jun 20:00 | 1° Grupo A | mejor 3° de C/E/F/H/I |
| L12 | 1 jul 11:00 | 1° Grupo L | mejor 3° de E/H/I/J/K |
| L13 | 3 jul 17:00 | 1° Grupo J | 2° Grupo H |
| L14 | 3 jul 13:00 | 2° Grupo D | 2° Grupo G |
| L15 | 2 jul 22:00 | 1° Grupo B | mejor 3° de E/F/G/I/J |
| L16 | 3 jul 20:30 | 1° Grupo K | mejor 3° de D/E/I/J/L |

### Restricción de fase predicha
| Situación | P ajustada |
|-----------|------------|
| 2+ rondas fuera de predicción | **3% / 97%** (forzado) |
| Exactamente en su ronda de eliminación | **8–18%** |
| ±1 ronda de tolerancia | ajuste **±30%** |
| Sin restricción | P base del clasificador binario |


In [14]:
import numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler

np.random.seed(2026)

# ══════════════════════════════════════════════════════════════════════════════
# 1. ENTRENAR CLASIFICADOR BINARIO PARA ELIMINATORIAS
# ══════════════════════════════════════════════════════════════════════════════
FEATURES_ELIM = [
    "wc_score", "elim_norm", "copa_norm", "ranking_score",
    "tm_cat", "copa_score", "rating_eafc26", "primera_vez"
]

# Equipos BLOQUEADOS: no pueden clasificar aunque aparezcan como 3° mejor
# Incluye debutantes + penalizados por OFC/nivel muy bajo
EQUIPOS_BLOQUEADOS = {"Curazao", "Haití", "Jordania", "Nueva Zelanda",
                       "Cabo Verde", "Uzbekistán"}

binary_pairs = []
hist = df_hist_model.copy()
pos_col = "posicion_final"

def get_feat_row(row):
    return np.array([
        row["wc_score"], row["elim_norm"], row["copa_norm"],
        row["ranking_score"], row["tm_cat"], row["copa_score"],
        row["rating_eafc26"] / 10.0, row["primera_vez"]
    ])

for torneo_offset, peso in [(0, 2.0), (32, 1.5)]:
    t_data = hist.iloc[torneo_offset:torneo_offset+32].reset_index(drop=True)
    avanzaron  = t_data[t_data[pos_col] <= 16]
    eliminados = t_data[t_data[pos_col] >  16]
    for _, w in avanzaron.iterrows():
        for _, l in eliminados.iterrows():
            xw   = get_feat_row(w)
            xl   = get_feat_row(l)
            diff = xw - xl
            binary_pairs.append((*diff, 1, peso))
            binary_pairs.append((*(-diff), 0, peso))

bp_cols    = [f"d_{f}" for f in FEATURES_ELIM] + ["y", "peso"]
df_binary  = pd.DataFrame(binary_pairs, columns=bp_cols)
X_bin      = df_binary[[f"d_{f}" for f in FEATURES_ELIM]].values
y_bin      = df_binary["y"].values
w_bin      = df_binary["peso"].values

scaler_bin = StandardScaler()
X_bin_sc   = scaler_bin.fit_transform(X_bin)

clf_params = dict(
    max_depth=4, n_estimators=200, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
    reg_alpha=0.1, reg_lambda=5.0, objective="binary:logistic",
    eval_metric="logloss", random_state=42, n_jobs=-1, verbosity=0,
)
clf = XGBClassifier(**clf_params)
clf.fit(X_bin_sc, y_bin, sample_weight=w_bin)

y_pred_proba = clf.predict_proba(X_bin_sc)[:, 1]
print("✅ Clasificador binario entrenado")
print(f"   Muestras : {len(X_bin_sc)}")
print(f"   ROC-AUC  : {roc_auc_score(y_bin, y_pred_proba):.4f}")
print(f"   Log-Loss : {log_loss(y_bin, y_pred_proba):.4f}\n")


# ══════════════════════════════════════════════════════════════════════════════
# 2. FUNCIÓN DE PREDICCIÓN CON RESTRICCIÓN DE FASE + BLEND XGB SCORE
# ══════════════════════════════════════════════════════════════════════════════
RONDA_ELIM  = {6:0, 5:1, 4:2, 3:3, 2:3, 1:4, 0:99, 7:0}
FASE_NOMBRE = {0:"Camp",1:"Sub",2:"3er",3:"4to",4:"QF",5:"R16",6:"R32",7:"Grps"}

def get_features_equipo(nombre):
    row = df_2026[df_2026["equipo"] == nombre]
    if row.empty:
        return np.zeros(len(FEATURES_ELIM))
    r = row.iloc[0]
    return np.array([r["wc_score"], r["elim_norm"], r["copa_norm"],
                     r["ranking_score"], r["tm_cat"], r["copa_score"],
                     r["rating_eafc26"] / 10.0, r["primera_vez"]])

def prob_xgb_score(teamA, teamB, escala=40.0):
    """Probabilidad basada directamente en el score XGBoost del regresor.
    Menor score = mejor equipo → si score_A < score_B, P(A) > 0.5."""
    sA = score_map.get(teamA, 50.0)
    sB = score_map.get(teamB, 50.0)
    return float(1 / (1 + np.exp((sA - sB) / escala)))

def prob_clasificador(teamA, teamB):
    """Probabilidad del clasificador binario (blend con score XGB)."""
    xA      = get_features_equipo(teamA)
    xB      = get_features_equipo(teamB)
    diff_sc = scaler_bin.transform((xA - xB).reshape(1, -1))
    p_clf   = float(clf.predict_proba(diff_sc)[0, 1])
    p_score = prob_xgb_score(teamA, teamB)
    # Blend 60% clasificador + 40% score regresor
    # Evita valores extremos (0%/100%) del clasificador solo
    return 0.60 * p_clf + 0.40 * p_score

def prob_restringida(teamA, teamB, ronda_actual):
    """
    Probabilidad final con restricción de fase ±1 ronda.
    
    fase_idx:   0=Camp · 1=Sub · 2=3er · 3=4to · 4=QF · 5=R16 · 6=R32 · 7=Grps
    RONDA_ELIM: ronda donde DEBE ser eliminado cada fase_idx
    
    Restricción:
      · 2+ rondas fuera de predicción → forzado 3% / 97%
      · Exactamente en su ronda      → p_win = 8–18%
      · ±1 ronda tolerancia          → ajuste ±30%
      · Sin restricción              → blend clasificador + score
    """
    fA = fase_predicha.get(teamA, 7)
    fB = fase_predicha.get(teamB, 7)
    rA = RONDA_ELIM.get(fA, 0)
    rB = RONDA_ELIM.get(fB, 0)

    # ── Restricción dura: 2+ rondas fuera ────────────────────────────────────
    if ronda_actual > rA + 1:
        return 0.03   # A llegó demasiado lejos → forzar derrota
    if ronda_actual > rB + 1:
        return 0.97   # B llegó demasiado lejos → forzar victoria A

    # ── Probabilidad base (blend) ─────────────────────────────────────────────
    p_base = prob_clasificador(teamA, teamB)

    # ── Restricción exacta: en su ronda de eliminación ───────────────────────
    if ronda_actual == rA and ronda_actual != rB:
        # A debe ser eliminado aquí → p_win muy baja
        return max(0.08, min(0.18, p_base * 0.25))
    if ronda_actual == rB and ronda_actual != rA:
        # B debe ser eliminado aquí → p_win muy alta
        return min(0.92, max(0.82, p_base * 0.25 + 0.75))

    # ── Restricción suave: ±1 ronda ───────────────────────────────────────────
    if ronda_actual == rA + 1 and ronda_actual <= rB:
        return max(0.12, p_base - 0.30)
    if ronda_actual == rB + 1 and ronda_actual <= rA:
        return min(0.88, p_base + 0.30)

    return p_base

def decidir_clasificado(teamA, teamB, ronda_actual):
    """Decide quién clasifica. Retorna (ganador, eliminado, prob_A)."""
    pA     = prob_restringida(teamA, teamB, ronda_actual)
    win_A  = pA >= 0.50          # gana quien supere el 50%
    winner = teamA if win_A else teamB
    loser  = teamB if win_A else teamA
    return winner, loser, pA


# ══════════════════════════════════════════════════════════════════════════════
# 3. CLASIFICADOS: PRIMEROS, SEGUNDOS Y MEJORES TERCEROS (SIN BLOQUEADOS)
# ══════════════════════════════════════════════════════════════════════════════
col_g = "grupo"
col_e = "equipo"
col_s = "score_adj" if "score_adj" in df_2026.columns else "score"

df_sorted = df_2026.sort_values([col_g, col_s]).copy()
df_sorted["rank_g"] = df_sorted.groupby(col_g)[col_s].rank(method="first").astype(int)

primeros = df_sorted[df_sorted["rank_g"] == 1].set_index(col_g)[col_e].to_dict()
segundos = df_sorted[df_sorted["rank_g"] == 2].set_index(col_g)[col_e].to_dict()

# Terceros: excluir bloqueados + ordenar por score (menor = mejor)
terceros_df = df_sorted[df_sorted["rank_g"] == 3].copy()
terceros_df = terceros_df[~terceros_df[col_e].isin(EQUIPOS_BLOQUEADOS)]
terceros_df = terceros_df.sort_values(col_s)  # menor score = mejor

mejores_terceros = [{"grupo": r[col_g], "equipo": r[col_e]}
                    for _, r in terceros_df.iterrows()]

# Si hay menos de 8 terceros disponibles (todos bloqueados en algún grupo),
# completar con los siguientes mejores del grupo (pos=4, no bloqueados)
if len(mejores_terceros) < 8:
    cuartos_df = df_sorted[df_sorted["rank_g"] == 4].copy()
    cuartos_df = cuartos_df[~cuartos_df[col_e].isin(EQUIPOS_BLOQUEADOS)]
    cuartos_df = cuartos_df.sort_values(col_s)
    extra = [{"grupo": r[col_g], "equipo": r[col_e]}
             for _, r in cuartos_df.iterrows()]
    mejores_terceros = (mejores_terceros + extra)[:8]

mejores_terceros = mejores_terceros[:8]

print("📊 Clasificados por grupo (score XGBoost):")
for g in sorted(primeros.keys()):
    t3_eq = next((t["equipo"] for t in mejores_terceros if t["grupo"]==g), "—")
    bl    = " [BLOQ]" if t3_eq in EQUIPOS_BLOQUEADOS else ""
    print(f"  Gr.{g}: 1°={primeros[g]:<22} 2°={segundos[g]:<22} 3°(MT?)={t3_eq}{bl}")

print()
print("✅ 8 Mejores terceros clasificados (sin bloqueados):")
for t in mejores_terceros:
    print(f"   Gr.{t['grupo']} {t['equipo']}")

print()
print(f"⛔ Equipos bloqueados (no pueden clasificar): {sorted(EQUIPOS_BLOQUEADOS)}")


# ══════════════════════════════════════════════════════════════════════════════
# 4. ASIGNACIÓN DE MEJORES TERCEROS + FIXTURE L01–L16
# ══════════════════════════════════════════════════════════════════════════════
def asignar_tercero(grupos_candidatos, excluir_grupo, ya_asignados):
    candidatos = [
        t for t in mejores_terceros
        if t["grupo"] in grupos_candidatos
        and t["grupo"] != excluir_grupo
        and t["equipo"] not in ya_asignados
        and t["equipo"] not in EQUIPOS_BLOQUEADOS
    ]
    if not candidatos:
        candidatos = [t for t in mejores_terceros
                      if t["equipo"] not in ya_asignados
                      and t["equipo"] not in EQUIPOS_BLOQUEADOS]
    if not candidatos:
        return "TBD"
    chosen = candidatos[np.random.randint(len(candidatos))]
    ya_asignados.add(chosen["equipo"])
    return chosen["equipo"]

asignados = set()
t_L01 = asignar_tercero(["A","B","C","D","F"], excluir_grupo="E", ya_asignados=asignados)
t_L02 = asignar_tercero(["C","D","F","G","H"], excluir_grupo="I", ya_asignados=asignados)
t_L07 = asignar_tercero(["B","E","F","I","J"], excluir_grupo="D", ya_asignados=asignados)
t_L08 = asignar_tercero(["A","E","H","I","J"], excluir_grupo="G", ya_asignados=asignados)
t_L11 = asignar_tercero(["C","E","F","H","I"], excluir_grupo="A", ya_asignados=asignados)
t_L12 = asignar_tercero(["E","H","I","J","K"], excluir_grupo="L", ya_asignados=asignados)
t_L15 = asignar_tercero(["E","F","G","I","J"], excluir_grupo="B", ya_asignados=asignados)
t_L16 = asignar_tercero(["D","E","I","J","L"], excluir_grupo="K", ya_asignados=asignados)

print("\n🔗 Asignación mejores terceros (sin bloqueados):")
for llave, eq, cab in [("L01",t_L01,"E"),("L02",t_L02,"I"),("L07",t_L07,"D"),
                        ("L08",t_L08,"G"),("L11",t_L11,"A"),("L12",t_L12,"L"),
                        ("L15",t_L15,"B"),("L16",t_L16,"K")]:
    print(f"   {llave}: 1°{cab} vs {eq}")

# Fixture oficial FIFA 2026
llaves_r32 = [
    ("L01", primeros["E"],  t_L01),          # 30jun 16:00
    ("L02", primeros["I"],  t_L02),          # 30jun 16:00
    ("L03", segundos["A"],  segundos["B"]),  # 28jun 14:00
    ("L04", primeros["F"],  segundos["C"]),  # 29jun 20:00
    ("L05", segundos["K"],  segundos["L"]),  # 2jul  18:00
    ("L06", primeros["H"],  segundos["J"]),  # 2jul  14:00
    ("L07", primeros["D"],  t_L07),          # 1jul  19:00
    ("L08", primeros["G"],  t_L08),          # 1jul  15:00
    ("L09", primeros["C"],  segundos["F"]),  # 29jun 12:00
    ("L10", segundos["E"],  segundos["I"]),  # 30jun 12:00
    ("L11", primeros["A"],  t_L11),          # 30jun 20:00
    ("L12", primeros["L"],  t_L12),          # 1jul  11:00
    ("L13", primeros["J"],  segundos["H"]),  # 3jul  17:00
    ("L14", segundos["D"],  segundos["G"]),  # 3jul  13:00
    ("L15", primeros["B"],  t_L15),          # 2jul  22:00
    ("L16", primeros["K"],  t_L16),          # 3jul  20:30
]


# ══════════════════════════════════════════════════════════════════════════════
# 5. SIMULAR TODAS LAS RONDAS (sin marcadores — solo ganador y P%)
# ══════════════════════════════════════════════════════════════════════════════
FECHAS_R32 = {
    "L01":"30 jun 16:00","L02":"30 jun 16:00","L03":"28 jun 14:00","L04":"29 jun 20:00",
    "L05":"2 jul 18:00", "L06":"2 jul 14:00", "L07":"1 jul 19:00", "L08":"1 jul 15:00",
    "L09":"29 jun 12:00","L10":"30 jun 12:00","L11":"30 jun 20:00","L12":"1 jul 11:00",
    "L13":"3 jul 17:00", "L14":"3 jul 13:00", "L15":"2 jul 22:00", "L16":"3 jul 20:30",
}

def simular_ronda(llaves, ronda):
    res = []
    for llave_id, tA, tB in llaves:
        ganador, eliminado, pA = decidir_clasificado(tA, tB, ronda)
        fA = FASE_NOMBRE.get(fase_predicha.get(tA, 7), "?")
        fB = FASE_NOMBRE.get(fase_predicha.get(tB, 7), "?")
        res.append({
            "llave": llave_id, "local": tA, "visitante": tB,
            "ganador": ganador, "eliminado": eliminado,
            "pred_local": fA, "pred_visit": fB,
            "prob_local": round(pA, 3),
            "fecha": FECHAS_R32.get(llave_id, ""),
        })
    return res

r32 = simular_ronda(llaves_r32, 0)

r16_llaves = [
    ("R16_A1", r32[0]["ganador"],  r32[1]["ganador"]),
    ("R16_A2", r32[2]["ganador"],  r32[3]["ganador"]),
    ("R16_A3", r32[4]["ganador"],  r32[5]["ganador"]),
    ("R16_A4", r32[6]["ganador"],  r32[7]["ganador"]),
    ("R16_B5", r32[8]["ganador"],  r32[9]["ganador"]),
    ("R16_B6", r32[10]["ganador"], r32[11]["ganador"]),
    ("R16_B7", r32[12]["ganador"], r32[13]["ganador"]),
    ("R16_B8", r32[14]["ganador"], r32[15]["ganador"]),
]
r16 = simular_ronda(r16_llaves, 1)

qf_llaves = [
    ("QF_A1", r16[0]["ganador"], r16[1]["ganador"]),
    ("QF_A2", r16[2]["ganador"], r16[3]["ganador"]),
    ("QF_B3", r16[4]["ganador"], r16[5]["ganador"]),
    ("QF_B4", r16[6]["ganador"], r16[7]["ganador"]),
]
qf = simular_ronda(qf_llaves, 2)

sf_llaves = [
    ("SF_1", qf[0]["ganador"], qf[1]["ganador"]),
    ("SF_2", qf[2]["ganador"], qf[3]["ganador"]),
]
sf = simular_ronda(sf_llaves, 3)

t3A, t3B         = sf[0]["eliminado"], sf[1]["eliminado"]
w3,  l3,  p3     = decidir_clasificado(t3A, t3B, 3)
champ, runner, pF = decidir_clasificado(sf[0]["ganador"], sf[1]["ganador"], 4)


# ══════════════════════════════════════════════════════════════════════════════
# 6. IMPRIMIR RESULTADOS — solo ganador y probabilidad, sin marcadores
# ══════════════════════════════════════════════════════════════════════════════
for nombre, ronda_res in [
    ("DIECISEISAVOS DE FINAL", r32),
    ("OCTAVOS DE FINAL",       r16),
    ("CUARTOS DE FINAL",       qf),
    ("SEMIFINALES",            sf),
]:
    print(f"\n{'='*70}")
    print(f"  {nombre}")
    print(f"{'='*70}")
    print(f"  {'Llave':<8} {'Local [Pred]':<28} {'Visitante [Pred]':<28} {'P(L)':>6}   Clasifica")
    print(f"  {'-'*67}")
    for r in ronda_res:
        arrow = "→" if r["ganador"] == r["local"] else "←"
        print(f"  {r['llave']:<8} "
              f"{r['local']:<22}[{r['pred_local']:>4}] "
              f"{r['visitante']:<22}[{r['pred_visit']:>4}] "
              f"{r['prob_local']:>6.0%}   {arrow} ✅ {r['ganador']}")

print(f"\n{'='*70}")
print(f"  3ER Y 4TO PUESTO")
print(f"{'='*70}")
print(f"  {t3A} vs {t3B}   P(1°)={p3:.0%}   🥉 {w3}  /  4° {l3}")

print(f"\n{'='*70}")
print(f"  GRAN FINAL 🏆")
print(f"{'='*70}")
print(f"  {sf[0]['ganador']} vs {sf[1]['ganador']}   P(1°)={pF:.0%}")
print(f"\n  🏆 CAMPEÓN    : {champ}")
print(f"  🥈 SUBCAMPEÓN : {runner}")
print(f"  🥉 3ER LUGAR  : {w3}")
print(f"  4TO LUGAR     : {l3}")
print(f"\n  Método: XGBoost Binario (60%) + Score Regresor (40%) | P≥50% clasifica")
print(f"  Bloqueados de clasificar: {sorted(EQUIPOS_BLOQUEADOS)}")


✅ Clasificador binario entrenado
   Muestras : 1024
   ROC-AUC  : 1.0000
   Log-Loss : 0.0030

📊 Clasificados por grupo (score XGBoost):
  Gr.A: 1°=Corea del Sur          2°=México                 3°(MT?)=Chequia
  Gr.B: 1°=Canadá                 2°=Suiza                  3°(MT?)=Qatar
  Gr.C: 1°=Brasil                 2°=Marruecos              3°(MT?)=—
  Gr.D: 1°=Estados Unidos         2°=Turquía                3°(MT?)=Australia
  Gr.E: 1°=Alemania               2°=Ecuador                3°(MT?)=—
  Gr.F: 1°=Países Bajos           2°=Suecia                 3°(MT?)=Japón
  Gr.G: 1°=Bélgica                2°=Irán                   3°(MT?)=—
  Gr.H: 1°=España                 2°=Uruguay                3°(MT?)=Arabia Saudita
  Gr.I: 1°=Francia                2°=Senegal                3°(MT?)=Noruega
  Gr.J: 1°=Argentina              2°=Jordania               3°(MT?)=Austria
  Gr.K: 1°=Portugal               2°=Colombia               3°(MT?)=—
  Gr.L: 1°=Inglaterra             2°=Croacia  

### Dashboard — Bracket visual completo (Fixture Oficial FIFA 2026)

El bracket sigue la **estructura oficial FIFA** con los dos lados del cuadro
(Lado A: llaves L01–L08, Lado B: llaves L09–L16). Los colores indican
la fase predicha por el **regresor XGBoost** y la probabilidad viene del
**clasificador binario** `binary:logistic`.


In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── PARCHE TEMPORAL ──────────────────────────────────────────────────────────
# Si ya tienes esta función en otra celda, ¡ejecútala! 
# Si no la tienes, aquí hay un sustituto temporal (devuelve 50% simulado) para que corra el script:
if 'prob_binaria_restringida' not in locals() and 'prob_binaria_restringida' not in globals():
    def prob_binaria_restringida(local, visitante, ajuste):
        return 0.50  # 50% por defecto. Reemplázalo con tu modelo XGBoost real.
# ─────────────────────────────────────────────────────────────────────────────

FASE_COLORS_PRED = {
    "Camp":"#FFD700","Sub":"#C0C0C0","3er":"#CD7F32","4to":"#4CAF50",
    "QF":"#2196F3","R16":"#7B1FA2","R32":"#9C27B0","Grps":"#607D8B",
}

FECHAS_R32 = {
    "L01":"30 jun 16:00","L02":"30 jun 16:00","L03":"28 jun 14:00","L04":"29 jun 20:00",
    "L05":"2 jul 18:00", "L06":"2 jul 14:00", "L07":"1 jul 19:00", "L08":"1 jul 15:00",
    "L09":"29 jun 12:00","L10":"30 jun 12:00","L11":"30 jun 20:00","L12":"1 jul 11:00",
    "L13":"3 jul 17:00", "L14":"3 jul 13:00", "L15":"2 jul 22:00", "L16":"3 jul 20:30",
}

def make_bracket_table(ronda_results, show_fecha=False):
    cols_hdr = ["<b>Llave</b>","<b>Local</b>","<b>Visitante</b>",
                "<b>P(Local/Visit)</b>","<b>Pred XGB</b>","<b>Clasificado ✅</b>"]
    if show_fecha:
        cols_hdr.insert(1, "<b>Fecha</b>")

    llaves    = [r["llave"]     for r in ronda_results]
    fechas    = [FECHAS_R32.get(r["llave"],"") for r in ronda_results]
    locales   = [r["local"]     for r in ronda_results]
    visits    = [r["visitante"] for r in ronda_results]
    probs     = [f"{r['prob_local']:.0%}" for r in ronda_results]
    pred_txt  = [f"[{r['pred_local']}]vs[{r['pred_visit']}]" for r in ronda_results]
    ganadores = [f"✅ {r['ganador']}" for r in ronda_results]

    fill_g = ["rgba(26,107,60,0.18)" if r["ganador"]==r["local"]
              else "rgba(183,28,28,0.10)" for r in ronda_results]
    fill_b = ["rgba(26,26,46,0.04)"] * len(ronda_results)

    vals = [llaves, locales, visits, probs, pred_txt, ganadores]
    aligns = ["center","left","left","center","center","left"]
    fills  = [fill_b, fill_b, fill_b, fill_b, fill_b, fill_g]

    if show_fecha:
        vals.insert(1, fechas)
        aligns.insert(1, "center")
        fills.insert(1, fill_b)

    return go.Table(
        header=dict(
            values=cols_hdr, fill_color="#1a1a2e",
            font=dict(color="white", size=10, family="Georgia, serif"),
            align=aligns, height=26,
        ),
        cells=dict(
            values=vals, fill_color=fills,
            font=dict(color="#1a1a2e", size=10, family="Georgia, serif"),
            align=aligns, height=23,
        ),
    )

subtitulos = [
    "<b>🔵 Dieciseisavos — Fixture Oficial FIFA 2026 (L01–L16 con fechas)</b>",
    "<b>🟣 Octavos — Lado A: G(L01-L04)·G(L05-L08) / Lado B: G(L09-L12)·G(L13-L16)</b>",
    "<b>🔷 Cuartos — QF_A1·QF_A2 (Lado A) / QF_B3·QF_B4 (Lado B)</b>",
    "<b>🟡 Semifinales — SF_1: G(QFA1)×G(QFA2) | SF_2: G(QFB3)×G(QFB4)</b>",
    "<b>🏆 Final & 3er Puesto</b>",
]

fig_bracket = make_subplots(
    rows=5, cols=1,
    subplot_titles=subtitulos,
    vertical_spacing=0.032,
    specs=[[{"type":"table"}]]*5,
)

fig_bracket.add_trace(make_bracket_table(r32, show_fecha=True), row=1, col=1)
fig_bracket.add_trace(make_bracket_table(r16),                  row=2, col=1)
fig_bracket.add_trace(make_bracket_table(qf),                   row=3, col=1)
fig_bracket.add_trace(make_bracket_table(sf),                   row=4, col=1)

final_data = [
    {"llave":"FINAL",
     "local":sf[0]["ganador"],"visitante":sf[1]["ganador"],
     "ganador":f"🏆 {champ}",
     "pred_local":FASE_NOMBRE.get(fase_predicha.get(sf[0]["ganador"],7),"?"),
     "pred_visit":FASE_NOMBRE.get(fase_predicha.get(sf[1]["ganador"],7),"?"),
     "prob_local":prob_binaria_restringida(sf[0]["ganador"],sf[1]["ganador"],4)},
    {"llave":"3°/4°",
     "local":t3A,"visitante":t3B,
     "ganador":f"🥉 {w3}",
     "pred_local":FASE_NOMBRE.get(fase_predicha.get(t3A,7),"?"),
     "pred_visit":FASE_NOMBRE.get(fase_predicha.get(t3B,7),"?"),
     "prob_local":prob_binaria_restringida(t3A,t3B,3)},
]
fig_bracket.add_trace(make_bracket_table(final_data), row=5, col=1)

fig_bracket.update_layout(
    title=dict(
        text=(
            f"<b>MUNDIAL 2026 — Bracket Oficial FIFA (Estructura L01–L16)</b><br>"
            f"<sup>🏆 {champ} · 🥈 {runner} · 🥉 {w3} · 4° {l3}<br>"
            f"Lado A: L01-L08 → R16_A1-A4 → QF_A1-A2 → SF_1 | "
            f"Lado B: L09-L16 → R16_B5-B8 → QF_B3-B4 → SF_2 → FINAL</sup>"
        ),
        font=dict(size=16, color="#1a1a2e"), x=0.01,
    ),
    height=2500,
    paper_bgcolor="white",
    font=dict(family="Georgia, serif"),
    margin=dict(l=20, r=20, t=130, b=30),
)
fig_bracket.show()

# ── Resumen ejecutivo ─────────────────────────────────────────────────────────
print(f"{chr(9552)*58}")
print(f"  RESUMEN EJECUTIVO — MUNDIAL 2026")
print(f"{chr(9552)*58}")
print(f"  Grupos       : XGBoost Regresor + score + ε~N(0,MAE)")
print(f"  Eliminatorias: XGBoost Clasificador Binario binary:logistic")
print(f"  Fixture      : Oficial FIFA 2026 (L01–L16)")
print(f"  Terceros     : Aleatorio entre candidatos de la llave")
print(f"  Restricción  : 2+rondas→3%/97% | exacta→8-18% | ±1→±30%")
print(f"  Partidos     : 72 grupos + 31 eliminatorias = 103 totales")
print(f"{chr(9472)*58}")
print(f"  🏆 Campeón     : {champ}")
print(f"  🥈 Subcampeón  : {runner}")
print(f"  🥉 3er lugar   : {w3}")
print(f"  4to lugar      : {l3}")
print(f"{chr(9552)*58}")


══════════════════════════════════════════════════════════
  RESUMEN EJECUTIVO — MUNDIAL 2026
══════════════════════════════════════════════════════════
  Grupos       : XGBoost Regresor + score + ε~N(0,MAE)
  Eliminatorias: XGBoost Clasificador Binario binary:logistic
  Fixture      : Oficial FIFA 2026 (L01–L16)
  Terceros     : Aleatorio entre candidatos de la llave
  Restricción  : 2+rondas→3%/97% | exacta→8-18% | ±1→±30%
  Partidos     : 72 grupos + 31 eliminatorias = 103 totales
──────────────────────────────────────────────────────────
  🏆 Campeón     : Inglaterra
  🥈 Subcampeón  : España
  🥉 3er lugar   : Portugal
  4to lugar      : Francia
══════════════════════════════════════════════════════════



<div style="background:linear-gradient(135deg,#0d1b2a,#1b2838);padding:32px;border-radius:12px;color:white;font-family:'Georgia',serif">
  <h3 style="margin:0 0 12px;font-size:18px">📊 Resumen del modelo — v8 (final)</h3>
  <ul style="color:#a8c4d8;line-height:2;font-size:13px">
    <li><b>Fase de grupos</b>: XGBoost Regresor (MAE) — score XGBoost + ε~N(0,MAE) · debutantes→4to</li>
    <li><b>Eliminatorias</b>: XGBoost Clasificador Binario (binary:logistic) — P(A elimina B)</li>
    <li><b>Fixture</b>: Estructura oficial FIFA 2026 (L01–L16 · Lado A + Lado B)</li>
    <li><b>Variables nuevas</b>: elim_norm (CSV Eliminatorias · Fuerza Final) + copa_norm (CSV Copas · Poder Final)</li>
    <li><b>Gain esperado</b>: wc_score > elim_norm > copa_norm > fifa_score > tm_cat > copa_score > eafc26/10 > primera_vez</li>
    <li><b>wc_score</b>: WC2022×2.0 + WC2018×1.5 (posición invertida)</li>
    <li><b>elim_fuerza</b>: Fuerza Base (suma ranking rivales) − Penalidad por posición final</li>
    <li><b>copa_poder</b>: Fuerza Base + Ajuste confederación (UEFA=+15, CONMEBOL=+7, etc.)</li>
    <li><b>tm_cat</b>: &lt;100M=0 · 100-400M=1 · 400-650M=2 · &gt;650M=3</li>
    <li><b>Entrenamiento</b>: WC2022 (peso 2.0) + WC2018 (peso 1.5) = 64 muestras con sample_weight</li>
  </ul>
  <div style="margin-top:16px;font-size:11px;color:#7eb3d8">
    v8 · Mayo 2026 · elim_fuerza y copa_poder desde CSVs del usuario
  </div>
</div>
